<a href="https://colab.research.google.com/github/Danielmalenga/AutoML/blob/main/Pipeline_Unificado_AutoML_CICIoT2023_3_Cenarios_10_Modelos_v3_CORRIGIDO_Pycaret.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline Unificado AutoML — CICIoT2023

## Execução sequencial dos 3 cenários e 10 frameworks

Este notebook consolida os códigos submetidos em um único fluxo:

1. classificação binária (`Benign` × `Malicious`);
2. classificação agrupada em 8 classes;
3. classificação multiclasse em 34 classes;
4. execução sequencial de Scikit-learn, PyCaret, TPOT, H2O, Auto-sklearn,
   FLAML, LazyPredict, Auto-PyTorch, AutoKeras e AutoGluon;
5. avaliação externa uniforme, usando **Macro-F1 como métrica principal**;
6. geração de métricas, relatórios, matrizes de confusão, rankings e logs.

Cada framework é instalado em ambiente Micromamba isolado para evitar conflitos
de versões. A falha de um framework é registrada e não interrompe os seguintes.


## 1. Montagem do Google Drive e diagnóstico do ambiente

In [4]:
# ============================================================
# 0. GOOGLE DRIVE E INFORMAÇÕES DO AMBIENTE
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import platform
import shutil
import sys

print("Python:", sys.version)
print("Sistema:", platform.platform())
print("Espaço em /content:", shutil.disk_usage("/content"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Sistema: Linux-6.6.122+-x86_64-with-glibc2.35
Espaço em /content: usage(total=120942624768, used=61830250496, free=59095597056)


## 2. Instalação das dependências do notebook orquestrador

In [5]:
# ============================================================
# 1. DEPENDÊNCIAS DO ORQUESTRADOR
# ============================================================

import subprocess
import sys

subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True,
)

subprocess.run(
    [
        "apt-get",
        "install",
        "-y",
        "-qq",
        "curl",
        "bzip2",
        "build-essential",
        "openjdk-17-jre-headless",
        "libgomp1",
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "duckdb>=1.1,<2",
        "pyarrow>=14,<20",
        "psutil",
    ],
    check=True,
)

print("Dependências do orquestrador instaladas.")


Dependências do orquestrador instaladas.


## 3. Configurações gerais, perfil e hiperparâmetros

In [6]:
# ============================================================
# 2. CONFIGURAÇÕES GERAIS, PERFIL E HIPERPARÂMETROS
# ============================================================

from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

DATASET_DIR = Path("/content/drive/MyDrive/Dataset/CSV")
RESULTS_DIR = Path(
    "/content/drive/MyDrive/CICIoT2023_Resultados/AutoML_Comparativo_Unificado"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FUSO_HORARIO = ZoneInfo("America/Sao_Paulo")
RUN_ID_GERAL = datetime.now(FUSO_HORARIO).strftime("%Y%m%d_%H%M%S")

#===================================================================
# Executa exatamente nesta ordem. Para teste, reduza a lista.
CENARIOS = ["binario", "agrupado8", "multiclasse34"]
#===================================================================

#===================================================================
# Seleção dos frameworks que serão executados.
# Use "todos" para executar a lista completa ou informe um framework específico.
EXECUTAR_FRAMEWORK = "todos"
#===================================================================
# Ordem recomendada: modelos rápidos/estáveis primeiro e pesados depois.
FRAMEWORKS = [
    "sklearn",
    "flaml",
    "lazypredict",
    "pycaret",
    "h2o",
    "tpot",
    "autogluon",
    "autokeras",
    "autopytorch",
    "autosklearn",
]

FRAMEWORKS_VALIDOS = set(FRAMEWORKS) | {"todos"}
EXECUTAR_FRAMEWORK = str(EXECUTAR_FRAMEWORK).lower().strip()
if EXECUTAR_FRAMEWORK not in FRAMEWORKS_VALIDOS:
    raise ValueError(
        f"EXECUTAR_FRAMEWORK inválido: {EXECUTAR_FRAMEWORK}. "
        f"Opções: {sorted(FRAMEWORKS_VALIDOS)}"
    )

# "rapido": valida o pipeline; "equilibrado": experimento recomendado;
# "completo": maior busca, tempo e consumo de disco/RAM.
#==========================================================================
PERFIL_EXECUCAO = "rapido"
#==========================================================================
PERFIS = {
    "rapido": {
        "MAX_ROWS_PER_CLASS": 2_000,
        "TIME_LIMIT": 600,
        "TPOT_GENERATIONS": 2,
        "TPOT_POPULATION_SIZE": 10,
        "H2O_MAX_MODELS": 10,
        "AUTOKERAS_MAX_TRIALS": 5,
        "AUTOKERAS_EPOCHS": 20,
    },
    "equilibrado": {
        "MAX_ROWS_PER_CLASS": 50_000,
        "TIME_LIMIT": 3_600,
        "TPOT_GENERATIONS": 5,
        "TPOT_POPULATION_SIZE": 20,
        "H2O_MAX_MODELS": 20,
        "AUTOKERAS_MAX_TRIALS": 10,
        "AUTOKERAS_EPOCHS": 50,
    },
    "completo": {
        "MAX_ROWS_PER_CLASS": None,
        "TIME_LIMIT": 10_800,
        "TPOT_GENERATIONS": 10,
        "TPOT_POPULATION_SIZE": 50,
        "H2O_MAX_MODELS": 40,
        "AUTOKERAS_MAX_TRIALS": 25,
        "AUTOKERAS_EPOCHS": 100,
    },
}

if PERFIL_EXECUCAO not in PERFIS:
    raise ValueError(f"PERFIL_EXECUCAO inválido: {PERFIL_EXECUCAO}")

P = PERFIS[PERFIL_EXECUCAO]
CHUNKSIZE = 200_000
MAX_FILES = None
MAX_ROWS_PER_FILE = None
MAX_ROWS_PER_CLASS = P["MAX_ROWS_PER_CLASS"]
IGNORE_MERGED_FILES = True
SELECAO_ARQUIVOS_BALANCEADA = True

TEST_SIZE = 0.20
VALIDATION_SPLIT = 0.20
RANDOM_STATE = 42
STRATIFY = True

# Métrica única de seleção e comparação.
OBJECTIVE = "f1_macro"
OPTIMIZE_METRIC = OBJECTIVE
EVAL_METRIC = OBJECTIVE

CARREGAR_MODELO_EXISTENTE = False
TREINAR_SE_NAO_EXISTIR = True
APAGAR_MODELO_ANTERIOR = False
REPROCESSAR_DADOS = True

# Um modelo por vez reduz competição por RAM e torna a comparação reproduzível.
N_JOBS = 1
TIME_LIMIT = P["TIME_LIMIT"]

# PyCaret
PYCARET_FOLDS = 5
PYCARET_INCLUDE_MODELS = ["lr", "dt", "rf", "et", "lightgbm", "xgboost"]

# TPOT
TPOT_GENERATIONS = P["TPOT_GENERATIONS"]
TPOT_POPULATION_SIZE = P["TPOT_POPULATION_SIZE"]
TPOT_CV = 5
TPOT_MAX_EVAL_TIME_MINS = 10
TPOT_EARLY_STOP = 5

# H2O
H2O_NTHREADS = 2
H2O_MAX_MEM_SIZE = "12G"
H2O_MAX_MODELS = P["H2O_MAX_MODELS"]
H2O_NFOLDS = 5
H2O_MAX_RUNTIME_PER_MODEL = 600
H2O_STOPPING_ROUNDS = 5
H2O_STOPPING_TOLERANCE = 1e-4
H2O_KEEP_CV_MODELS = False
H2O_KEEP_CV_PREDICTIONS = False
H2O_KEEP_CV_FOLD_ASSIGNMENT = False

# Auto-sklearn e Auto-PyTorch
AUTOSKLEARN_MEMORY_MB = 12_000
AUTOSKLEARN_PER_RUN_SECS = 600
AUTOSKLEARN_ENSEMBLE_SIZE = 25
AUTOSKLEARN_ENSEMBLE_NBEST = 25
AUTOSKLEARN_MAX_MODELS_ON_DISC = 30

AUTOPYTORCH_MEMORY_MB = 12_000
AUTOPYTORCH_N_THREADS = 2
AUTOPYTORCH_ENSEMBLE_SIZE = 20
AUTOPYTORCH_ENSEMBLE_NBEST = 20
AUTOPYTORCH_MAX_MODELS_ON_DISC = 30
AUTOPYTORCH_FUNC_EVAL_SECS = 600
AUTOPYTORCH_MIN_EPOCHS = 10
AUTOPYTORCH_MAX_EPOCHS = 50
AUTOPYTORCH_ENABLE_TRADITIONAL = True

# FLAML, LazyPredict, AutoKeras e AutoGluon
FLAML_ESTIMATOR_LIST = ["lgbm", "xgboost", "rf", "extra_tree", "lrl1"]
LAZYPREDICT_MODE = "stable"
AUTOKERAS_MAX_TRIALS = P["AUTOKERAS_MAX_TRIALS"]
AUTOKERAS_EPOCHS = P["AUTOKERAS_EPOCHS"]
AUTOKERAS_PATIENCE = 7
AUTOKERAS_OBJECTIVE = "val_macro_f1"
AUTOGLUON_PRESETS = "best_quality"
AUTOGLUON_NUM_GPUS = 0

LIMPAR_PARTES_APOS_SPLIT = True
REUTILIZAR_AMBIENTES = True
REMOVER_AMBIENTE_APOS_SUCESSO = False
MANTER_AMBIENTE_SE_ERRO = True
CONTINUAR_APOS_ERRO = True

WORK_DIR = Path("/content/CICIoT2023_AutoML_Pipeline")
WORK_DIR.mkdir(parents=True, exist_ok=True)
RUNNER_SCRIPT = WORK_DIR / "automl_runner.py"
DUCKDB_DATABASE = WORK_DIR / "preparacao.duckdb"
DUCKDB_TEMP_DIR = WORK_DIR / "duckdb_temp"
DUCKDB_MEMORY_LIMIT = "24GB"
DUCKDB_THREADS = 4

print("Perfil:", PERFIL_EXECUCAO)
print("Cenários:", CENARIOS)
print("Frameworks disponíveis:", FRAMEWORKS)
print("Seleção de execução:", EXECUTAR_FRAMEWORK)
print("Métrica principal:", OBJECTIVE)


Perfil: rapido
Cenários: ['binario', 'agrupado8', 'multiclasse34']
Frameworks disponíveis: ['sklearn', 'flaml', 'lazypredict', 'pycaret', 'h2o', 'tpot', 'autogluon', 'autokeras', 'autopytorch', 'autosklearn']
Seleção de execução: todos
Métrica principal: f1_macro


## 4. Classes, rótulos e caminhos dos três cenários

In [7]:
# ============================================================
# 3. DEFINIÇÃO E CONFIGURAÇÃO DOS CENÁRIOS
# ============================================================

LABEL_ORIGEM = "label_original"
CLASSE_BENIGNA = "Benign"
CLASSE_MALICIOSA = "Malicious"
CLASSES_BINARIAS = ["Benign", "Malicious"]
CLASSES_AGRUPADAS8 = [
    "Benign", "BruteForce", "DDoS", "DoS",
    "Mirai", "Recon", "Spoofing", "Web",
]
CLASSES_MULTICLASSE34 = [
    "Backdoor_Malware", "BenignTraffic", "BrowserHijacking",
    "CommandInjection", "DDoS-ACK_Fragmentation", "DDoS-HTTP_Flood",
    "DDoS-ICMP_Flood", "DDoS-ICMP_Fragmentation", "DDoS-PSHACK_Flood",
    "DDoS-RSTFINFlood", "DDoS-SYN_Flood", "DDoS-SlowLoris",
    "DDoS-SynonymousIP_Flood", "DDoS-TCP_Flood", "DDoS-UDP_Flood",
    "DDoS-UDP_Fragmentation", "DictionaryBruteForce", "DNS_Spoofing",
    "DoS-HTTP_Flood", "DoS-SYN_Flood", "DoS-TCP_Flood",
    "DoS-UDP_Flood", "MITM-ArpSpoofing", "Mirai-greeth_flood",
    "Mirai-greip_flood", "Mirai-udpplain", "Recon-HostDiscovery",
    "Recon-OSScan", "Recon-PingSweep", "Recon-PortScan", "SqlInjection",
    "Uploading_Attack", "VulnerabilityScan", "XSS",
]


def configurar_cenario(cenario):
    """Atualiza os caminhos e metadados usados pelas funções compartilhadas."""
    global CENARIO, LABEL_FINAL, CLASSES_FINAIS, PROBLEM_TYPE, NUM_CLASSES
    global RUN_RESULTS_DIR, PREPARED_DIR, PARTS_DIR
    global TRAIN_PARQUET, TEST_PARQUET, DATASET_INFO_JSON
    global CONFIG_JSON, LOGS_DIR

    CENARIO = cenario
    if cenario == "binario":
        LABEL_FINAL = "label_binary"
        CLASSES_FINAIS = list(CLASSES_BINARIAS)
        PROBLEM_TYPE = "binary"
    elif cenario == "agrupado8":
        LABEL_FINAL = "label_grouped"
        CLASSES_FINAIS = list(CLASSES_AGRUPADAS8)
        PROBLEM_TYPE = "multiclass"
    elif cenario == "multiclasse34":
        LABEL_FINAL = "label_multiclass"
        CLASSES_FINAIS = list(CLASSES_MULTICLASSE34)
        PROBLEM_TYPE = "multiclass"
    else:
        raise ValueError(f"Cenário inválido: {cenario}")

    NUM_CLASSES = len(CLASSES_FINAIS)
    RUN_RESULTS_DIR = RESULTS_DIR / RUN_ID_GERAL / CENARIO
    PREPARED_DIR = RESULTS_DIR / "_dados_preparados" / CENARIO
    PARTS_DIR = PREPARED_DIR / "_partes_temporarias"
    TRAIN_PARQUET = PREPARED_DIR / "train_data.parquet"
    TEST_PARQUET = PREPARED_DIR / "test_data.parquet"
    DATASET_INFO_JSON = PREPARED_DIR / "dataset_info.json"
    CONFIG_JSON = WORK_DIR / f"config_{CENARIO}.json"
    LOGS_DIR = RUN_RESULTS_DIR / "_logs"

    for pasta in [RUN_RESULTS_DIR, PREPARED_DIR, PARTS_DIR, LOGS_DIR]:
        pasta.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 100)
    print(f"CENÁRIO: {CENARIO} | CLASSES: {NUM_CLASSES} | ALVO: {LABEL_FINAL}")
    print("=" * 100)


## 5. Funções de preparação out-of-core do dataset

In [8]:
# ============================================================
# 4. PREPARAÇÃO OUT-OF-CORE DO CICIoT2023
# ============================================================

import gc
import hashlib
import json
import math
import os
import re
import shutil
import time
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd


def detectar_coluna_rotulo(columns):
    candidatos = [
        "label", "Label", "Attack", "attack", "Class", "class",
        "category", "Category", "type", "Type",
        "label_multiclass", "label_grouped", "label_binary",
    ]
    for col in candidatos:
        if col in columns:
            return col
    return None


def extrair_rotulo_do_nome_arquivo(nome_arquivo):
    nome = str(nome_arquivo)
    if nome.lower().endswith(".csv"):
        nome = nome[:-4]
    if nome.lower().endswith(".pcap"):
        nome = nome[:-5]
    return nome


# ============================================================
# NORMALIZAÇÃO DOS RÓTULOS ORIGINAIS
# ============================================================

def normalizar_rotulo_original(valor):
    """Normaliza variações conhecidas sem alterar os nomes oficiais."""
    if valor is None or (isinstance(valor, float) and np.isnan(valor)):
        return None

    valor = str(valor).strip()
    valor = re.sub(r"\s+", " ", valor)
    chave = re.sub(r"[\s_-]+", "", valor).lower()

    correcoes_por_chave = {
        "benign": "BenignTraffic",
        "benigntraffic": "BenignTraffic",
        "ddoshttpflood": "DDoS-HTTP_Flood",
        "doshttpflood": "DoS-HTTP_Flood",
        "ddosslowloris": "DDoS-SlowLoris",
        "sqlinjection": "SqlInjection",
        "dictionarybruteforce": "DictionaryBruteForce",
        "dnsspoofing": "DNS_Spoofing",
        "mitmarpspoofing": "MITM-ArpSpoofing",
    }
    return correcoes_por_chave.get(chave, valor)


# ============================================================
# CENÁRIO BINÁRIO
# ============================================================

def converter_para_binario(valor):
    valor = normalizar_rotulo_original(valor)

    # Rótulos nulos não podem chamar .lower().
    if valor is None:
        return None

    if "benign" in valor.lower():
        return "Benign"

    return "Malicious"


# ============================================================
# CENÁRIO AGRUPADO EM 8 CLASSES
# ============================================================

def converter_para_agrupado8(valor):
    valor = normalizar_rotulo_original(valor)

    # Rótulos nulos não podem chamar .lower().
    if valor is None:
        return None

    valor_lower = valor.lower()

    if "benign" in valor_lower:
        return "Benign"

    if "dictionarybruteforce" in valor_lower:
        return "BruteForce"

    if valor_lower.startswith("ddos"):
        return "DDoS"

    if valor_lower.startswith("dos"):
        return "DoS"

    if valor_lower.startswith("mirai"):
        return "Mirai"

    if (
        valor_lower.startswith("recon")
        or "vulnerabilityscan" in valor_lower
    ):
        return "Recon"

    if (
        "dns_spoofing" in valor_lower
        or "mitm-arpspoofing" in valor_lower
    ):
        return "Spoofing"

    ataques_web = [
        "backdoor_malware",
        "browserhijacking",
        "commandinjection",
        "sqlinjection",
        "uploading_attack",
        "xss",
    ]

    if any(
        ataque in valor_lower
        for ataque in ataques_web
    ):
        return "Web"

    return None


# ============================================================
# CENÁRIO MULTICLASSE DE 34 CLASSES
# ============================================================

def converter_para_multiclasse34(valor):
    valor = normalizar_rotulo_original(valor)

    # Mantém a mesma proteção explícita em todos os conversores.
    if valor is None:
        return None

    if valor in CLASSES_MULTICLASSE34:
        return valor

    return None


# ============================================================
# SELEÇÃO AUTOMÁTICA DO CONVERSOR
# ============================================================

def converter_rotulo(valor):
    if CENARIO == "binario":
        return converter_para_binario(valor)

    if CENARIO == "agrupado8":
        return converter_para_agrupado8(valor)

    if CENARIO == "multiclasse34":
        return converter_para_multiclasse34(valor)

    raise ValueError(
        f"Cenário inválido: {CENARIO}"
    )





def listar_csvs():
    if not DATASET_DIR.exists():
        raise FileNotFoundError(
            f"Diretório do dataset não encontrado: {DATASET_DIR}"
        )

    arquivos = sorted(DATASET_DIR.rglob("*.csv"))
    if IGNORE_MERGED_FILES:
        arquivos = [
            a for a in arquivos
            if "merged" not in a.name.lower()
        ]

    if MAX_FILES is not None:
        limite = int(MAX_FILES)
        if limite <= 0:
            raise ValueError("MAX_FILES deve ser positivo ou None.")

        if SELECAO_ARQUIVOS_BALANCEADA:
            benignos = [
                a for a in arquivos
                if "benign" in a.name.lower()
            ]
            maliciosos = [
                a for a in arquivos
                if "benign" not in a.name.lower()
            ]

            rng = np.random.default_rng(RANDOM_STATE)
            rng.shuffle(benignos)
            rng.shuffle(maliciosos)

            n_benignos = min(
                len(benignos),
                max(1, limite // 2),
            )
            n_maliciosos = min(
                len(maliciosos),
                limite - n_benignos,
            )

            selecionados = (
                benignos[:n_benignos]
                + maliciosos[:n_maliciosos]
            )

            # Completa o limite caso um dos grupos tenha poucos arquivos.
            if len(selecionados) < limite:
                usados = set(selecionados)
                restantes = [
                    a for a in arquivos
                    if a not in usados
                ]
                rng.shuffle(restantes)
                selecionados.extend(
                    restantes[: limite - len(selecionados)]
                )

            arquivos = sorted(
                selecionados,
                key=lambda p: p.name.lower(),
            )
        else:
            arquivos = arquivos[:limite]

    if not arquivos:
        raise FileNotFoundError(
            f"Nenhum CSV encontrado em: {DATASET_DIR}"
        )

    return arquivos

def assinatura_preprocessamento():
    return {
        "dataset_dir": str(DATASET_DIR),
        "chunksize": CHUNKSIZE,
        "max_files": MAX_FILES,
        "max_rows_per_file": MAX_ROWS_PER_FILE,
        "max_rows_per_class": MAX_ROWS_PER_CLASS,
        "selecao_arquivos_balanceada": SELECAO_ARQUIVOS_BALANCEADA,
        "ignore_merged_files": IGNORE_MERGED_FILES,
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "stratify": STRATIFY,
        "label_final": LABEL_FINAL,
    }


def cache_preparado_valido():
    if REPROCESSAR_DADOS:
        return False
    if not TRAIN_PARQUET.exists() or not TEST_PARQUET.exists() or not DATASET_INFO_JSON.exists():
        return False
    try:
        info = json.loads(DATASET_INFO_JSON.read_text(encoding="utf-8"))
        return info.get("assinatura") == assinatura_preprocessamento()
    except Exception:
        return False


def limpar_colunas_de_vazamento(chunk, coluna_rotulo_detectada):
    candidatos = {
        "label", "Label", "Attack", "attack", "Class", "class",
        "category", "Category", "type", "Type",
        "label_multiclass", "label_grouped", "label_binary",
        LABEL_ORIGEM, "source_file", "Source_File", "filename", "Filename",
    }
    candidatos.discard(LABEL_FINAL)
    if coluna_rotulo_detectada:
        candidatos.add(coluna_rotulo_detectada)
    return chunk.drop(columns=[c for c in candidatos if c in chunk.columns], errors="ignore")


def processar_csvs_para_partes():
    arquivos = listar_csvs()
    print(f"Arquivos selecionados: {len(arquivos)}")

    if PARTS_DIR.exists():
        shutil.rmtree(PARTS_DIR)
    PARTS_DIR.mkdir(parents=True, exist_ok=True)

    total_linhas = 0
    total_partes = 0
    contagem_classes = {
    classe: 0
    for classe in CLASSES_FINAIS
}
    colunas_observadas = set()
    erros = []
    inicio = time.time()

    for file_idx, arquivo in enumerate(arquivos, start=1):
        print(f"[{file_idx}/{len(arquivos)}] {arquivo.name}")
        rotulo_fallback = extrair_rotulo_do_nome_arquivo(arquivo.name)
        linhas_arquivo = 0

        try:
            leitor = pd.read_csv(
                arquivo,
                chunksize=CHUNKSIZE,
                nrows=MAX_ROWS_PER_FILE,
                low_memory=False,
            )

            for chunk_idx, chunk in enumerate(leitor):
                if chunk.empty:
                    continue

                chunk.columns = chunk.columns.astype(str).str.strip()
                coluna_rotulo = detectar_coluna_rotulo(chunk.columns)

                rotulo_arquivo = normalizar_rotulo_original(rotulo_fallback)
                if coluna_rotulo is not None:
                    rotulo_original = chunk[coluna_rotulo]
                    rotulo_convertido = rotulo_original.map(converter_rotulo)
                    fallback_convertido = converter_rotulo(rotulo_arquivo)
                    if fallback_convertido in CLASSES_FINAIS:
                        rotulo_convertido = rotulo_convertido.fillna(fallback_convertido)
                else:
                    rotulo_original = pd.Series(
                        rotulo_arquivo, index=chunk.index, dtype="object"
                    )
                    rotulo_convertido = rotulo_original.map(converter_rotulo)

                chunk[LABEL_FINAL] = rotulo_convertido
                chunk = chunk[chunk[LABEL_FINAL].isin(CLASSES_FINAIS)].copy()
              # Remove colunas que poderiam causar vazamento de dados.
                chunk = limpar_colunas_de_vazamento(chunk, coluna_rotulo)

                # Converte os atributos do CICIoT2023 para float32.
                feature_cols = [c for c in chunk.columns if c != LABEL_FINAL]
                for col in feature_cols:
                    chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

                chunk[feature_cols] = chunk[feature_cols].replace(
                    [np.inf, -np.inf],
                    np.nan,
                )
                for col in feature_cols:
                    try:
                        chunk[col] = chunk[col].astype(np.float32)
                    except Exception:
                        pass

                # Identificador numérico reproduzível para o split em disco.
                inicio_id = linhas_arquivo
                fim_id = linhas_arquivo + len(chunk)
                base = np.uint64(file_idx) * np.uint64(10**12)
                chunk["__row_id"] = base + np.arange(
                    inicio_id,
                    fim_id,
                    dtype=np.uint64,
                )

                linhas_arquivo += len(chunk)
                total_linhas += len(chunk)
                total_partes += 1
                colunas_observadas.update(chunk.columns)

                counts = chunk[LABEL_FINAL].value_counts()
                for classe in CLASSES_FINAIS:
                    contagem_classes[classe] += int(counts.get(classe, 0))

                parte_path = PARTS_DIR / f"part_{total_partes:06d}.parquet"
                chunk.to_parquet(
                    parte_path,
                    index=False,
                    engine="pyarrow",
                    compression="snappy",
                )

                del chunk
                gc.collect()

        except Exception as exc:
            erros.append({"arquivo": arquivo.name, "erro": repr(exc)})
            print(f"  ERRO: {exc}")

    if total_partes == 0:
        raise RuntimeError("Nenhuma parte Parquet foi criada.")

    classes_insuficientes = {
        classe: int(contagem_classes.get(classe, 0))
        for classe in CLASSES_FINAIS
        if int(contagem_classes.get(classe, 0)) < 2
    }

    diagnostico_classes = {
        "cenario": CENARIO,
        "classes_esperadas": list(CLASSES_FINAIS),
        "contagem_antes_split": contagem_classes,
        "classes_insuficientes": classes_insuficientes,
        "erros_leitura": erros,
    }
    (PREPARED_DIR / "diagnostico_classes.json").write_text(
        json.dumps(diagnostico_classes, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

    if classes_insuficientes:
        raise ValueError(
            "Classes ausentes ou com menos de duas amostras antes do split: "
            f"{classes_insuficientes}. Consulte "
            f"{PREPARED_DIR / 'diagnostico_classes.json'}"
        )

    resumo = {
        "arquivos_processados": len(arquivos),
        "partes_criadas": total_partes,
        "linhas_antes_do_split": total_linhas,
        "classes_antes_do_split": contagem_classes,
        "colunas_observadas": sorted(colunas_observadas),
        "erros_leitura": erros,
        "duracao_processamento_segundos": time.time() - inicio,
    }
    return resumo


def criar_split_estratificado_em_disco():
    """
    Cria treino e teste sem concatenar todos os CSVs na memória.

    Quando STRATIFY=True, a ordenação determinística por hash é feita
    separadamente dentro de cada classe. Isso preserva exatamente, salvo
    arredondamento inteiro, TEST_SIZE de cada classe.
    """
    PREPARED_DIR.mkdir(parents=True, exist_ok=True)
    DUCKDB_TEMP_DIR.mkdir(parents=True, exist_ok=True)

    if TRAIN_PARQUET.exists():
        TRAIN_PARQUET.unlink()
    if TEST_PARQUET.exists():
        TEST_PARQUET.unlink()
    if DUCKDB_DATABASE.exists():
        DUCKDB_DATABASE.unlink()

    con = duckdb.connect(str(DUCKDB_DATABASE))
    try:
        con.execute(f"PRAGMA temp_directory='{DUCKDB_TEMP_DIR.as_posix()}';")
        con.execute(f"PRAGMA memory_limit='{DUCKDB_MEMORY_LIMIT}';")
        con.execute(f"PRAGMA threads={DUCKDB_THREADS};")

        partes_glob = (PARTS_DIR / "*.parquet").as_posix()
        limite_sql = ""
        if MAX_ROWS_PER_CLASS is not None:
            limite_sql = f"WHERE class_rank <= {int(MAX_ROWS_PER_CLASS)}"

        if STRATIFY:
            partition_expr = LABEL_FINAL
        else:
            partition_expr = "'ALL'"

        print("Criando tabela temporária ranqueada no DuckDB...")
        con.execute(
            f"""
            CREATE TABLE dados_ranqueados AS
            WITH origem AS (
                SELECT *
                FROM read_parquet('{partes_glob}', union_by_name=true)
            ),
            limitacao AS (
                SELECT
                    *,
                    row_number() OVER (
                        PARTITION BY {LABEL_FINAL}
                        ORDER BY hash(__row_id, {RANDOM_STATE})
                    ) AS class_rank
                FROM origem
            ),
            selecionados AS (
                SELECT *
                FROM limitacao
                {limite_sql}
            )
            SELECT
                * EXCLUDE(class_rank),
                row_number() OVER (
                    PARTITION BY {partition_expr}
                    ORDER BY hash(__row_id, {RANDOM_STATE + 1})
                ) AS split_rank,
                count(*) OVER (
                    PARTITION BY {partition_expr}
                ) AS split_n
            FROM selecionados
            """
        )

        test_path = TEST_PARQUET.as_posix()
        train_path = TRAIN_PARQUET.as_posix()

        print("Exportando conjunto de teste...")
        con.execute(
            f"""
            COPY (
                SELECT * EXCLUDE(split_rank, split_n, __row_id)
                FROM dados_ranqueados
                WHERE split_rank <= greatest(1, floor(split_n * {float(TEST_SIZE)}))
            )
            TO '{test_path}'
            (FORMAT PARQUET, COMPRESSION ZSTD)
            """
        )

        print("Exportando conjunto de treinamento...")
        con.execute(
            f"""
            COPY (
                SELECT * EXCLUDE(split_rank, split_n, __row_id)
                FROM dados_ranqueados
                WHERE split_rank > greatest(1, floor(split_n * {float(TEST_SIZE)}))
            )
            TO '{train_path}'
            (FORMAT PARQUET, COMPRESSION ZSTD)
            """
        )

        treino_dist = con.execute(
            f"""
            SELECT {LABEL_FINAL}, count(*) AS n
            FROM read_parquet('{train_path}')
            GROUP BY {LABEL_FINAL}
            ORDER BY {LABEL_FINAL}
            """
        ).fetchdf()

        teste_dist = con.execute(
            f"""
            SELECT {LABEL_FINAL}, count(*) AS n
            FROM read_parquet('{test_path}')
            GROUP BY {LABEL_FINAL}
            ORDER BY {LABEL_FINAL}
            """
        ).fetchdf()

        total_treino = int(treino_dist["n"].sum())
        total_teste = int(teste_dist["n"].sum())

        return {
            "train_shape_rows": total_treino,
            "test_shape_rows": total_teste,
            "train_distribution": dict(zip(treino_dist[LABEL_FINAL], treino_dist["n"].astype(int))),
            "test_distribution": dict(zip(teste_dist[LABEL_FINAL], teste_dist["n"].astype(int))),
        }
    finally:
        con.close()


def preparar_dataset():
    if cache_preparado_valido():
        print("Split preparado encontrado e compatível. Reutilizando:")
        print(TRAIN_PARQUET)
        print(TEST_PARQUET)
        return json.loads(DATASET_INFO_JSON.read_text(encoding="utf-8"))

    resumo_partes = processar_csvs_para_partes()
    resumo_split = criar_split_estratificado_em_disco()

    info = {
        "assinatura": assinatura_preprocessamento(),
        "resumo_partes": resumo_partes,
        "resumo_split": resumo_split,
        "train_parquet": str(TRAIN_PARQUET),
        "test_parquet": str(TEST_PARQUET),
        "criado_em": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    DATASET_INFO_JSON.write_text(
        json.dumps(info, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

    if LIMPAR_PARTES_APOS_SPLIT and PARTS_DIR.exists():
        shutil.rmtree(PARTS_DIR)
    if DUCKDB_DATABASE.exists():
        DUCKDB_DATABASE.unlink()
    if DUCKDB_TEMP_DIR.exists():
        shutil.rmtree(DUCKDB_TEMP_DIR, ignore_errors=True)

    print(json.dumps(resumo_split, indent=2, ensure_ascii=False))
    return info

## 6. Preparação, divisão estratificada e validação de classes

In [9]:
def preparar_e_validar_cenario():
    global dataset_info, resumo_split
    # ============================================================
    # 5. PREPARAR E DIVIDIR O DATASET
    # ============================================================

    dataset_info = preparar_dataset()
    print("Treino:", TRAIN_PARQUET)
    print("Teste :", TEST_PARQUET)
    # ============================================================
    # DIAGNÓSTICO DA PREPARAÇÃO
    # ============================================================

    print(
        "MAX_FILES usado:",
        dataset_info["assinatura"]["max_files"],
    )

    print(
        "Partes criadas:",
        dataset_info["resumo_partes"]["partes_criadas"],
    )

    print(
        "Classes antes do split:",
        dataset_info["resumo_partes"]["classes_antes_do_split"],
    )

    print(
        "Erros de leitura:",
        dataset_info["resumo_partes"]["erros_leitura"],
    )
    # ============================================================
    # 5.1 VALIDAR AS CLASSES GERADAS
    # ============================================================

    if "dataset_info" not in globals():
        raise NameError(
            "A variável 'dataset_info' não existe. "
            "Execute primeiro a célula 5."
        )

    # Extrai o resumo da divisão retornado por preparar_dataset().
    resumo_split = dataset_info["resumo_split"]

    classes_treino = set(
        resumo_split["train_distribution"].keys()
    )

    classes_teste = set(
        resumo_split["test_distribution"].keys()
    )

    classes_esperadas = set(CLASSES_FINAIS)

    classes_ausentes_treino = (
        classes_esperadas - classes_treino
    )

    classes_ausentes_teste = (
        classes_esperadas - classes_teste
    )

    print("=" * 60)
    print("VALIDAÇÃO DAS CLASSES")
    print("=" * 60)

    print("Cenário:", CENARIO)
    print("Classes esperadas:", len(classes_esperadas))
    print("Classes no treinamento:", len(classes_treino))
    print("Classes no teste:", len(classes_teste))

    print(
        "\nDistribuição do treinamento:",
        resumo_split["train_distribution"],
    )

    print(
        "\nDistribuição do teste:",
        resumo_split["test_distribution"],
    )

    if classes_ausentes_treino:
        raise ValueError(
            "Classes ausentes no treinamento: "
            f"{sorted(classes_ausentes_treino)}"
        )

    if classes_ausentes_teste:
        raise ValueError(
            "Classes ausentes no teste: "
            f"{sorted(classes_ausentes_teste)}"
        )

    print("\nValidação concluída: todas as classes estão presentes.")


## 7. Especificação dos ambientes isolados por framework

In [10]:
# ============================================================
# 6. AMBIENTES ISOLADOS POR FRAMEWORK
# ============================================================

import os
import subprocess
import sys
from pathlib import Path


# ============================================================
# CAMINHOS DO MICROMAMBA
# ============================================================

# Mantidos como objetos Path para permitir o uso do operador "/".
MAMBA_ROOT_PREFIX = Path("/content/micromamba")
MICROMAMBA = Path("/content/bin/micromamba")

# Variáveis de ambiente precisam receber valores do tipo string.
os.environ["MAMBA_ROOT_PREFIX"] = str(MAMBA_ROOT_PREFIX)
os.environ["PIP_NO_CACHE_DIR"] = "1"

FRAMEWORK_ENV_SPECS = {
    "sklearn": {
        "python": "3.11",
        "conda": ["pip"],
        "pip": [
            "numpy==1.26.4", "pandas==2.1.4",
            "scikit-learn==1.5.2", "matplotlib==3.8.4",
            "pyarrow==15.0.2", "joblib", "psutil",
        ],
        "check": (
            "import sklearn; "
            "from sklearn.ensemble import ExtraTreesClassifier; "
            "print('Scikit-learn:', sklearn.__version__)"
        ),
    },
  "pycaret": {
    "python": "3.10",

    # Runtime C/C++ compatível com as dependências
    # binárias utilizadas pelo PyCaret e pelo XGBoost.
    "conda": [
        "pip",
        "libstdcxx>=14",
        "libstdcxx-ng>=14",
        "libgcc>=14",
        "libgcc-ng>=14",
    ],

    "pip": [
        "numpy==1.26.4",
        "pandas==2.1.4",
        "scikit-learn==1.4.2",
        "matplotlib==3.7.5",
        "pyarrow==15.0.2",
        "joblib",
        "psutil",

        # Framework AutoML.
        "pycaret==3.3.2",

        # Necessário para disponibilizar o estimador
        # identificado como 'xgboost' no compare_models().
        "xgboost==2.1.4",
    ],

    "check": (
        "import pycaret; "
        "import sklearn; "
        "import xgboost; "
        "from pycaret.classification import setup; "
        "print('PyCaret:', pycaret.__version__); "
        "print('Scikit-learn:', sklearn.__version__); "
        "print('XGBoost:', xgboost.__version__)"
    ),
},
    "tpot": {
        "python": "3.10",
        "conda": ["pip"],
        "pip": [
            "numpy==1.26.4",
            "pandas==2.1.4",
            "scikit-learn==1.5.2",
            "matplotlib==3.8.4",
            "pyarrow==15.0.2",
            "joblib",
            "psutil",
            # TPOT 0.12.2/stopit ainda importa pkg_resources.
            # Setuptools 81+ removeu esse módulo.
            "setuptools==80.9.0",
            "tpot==0.12.2",
            "xgboost==2.1.4",
            "lightgbm==4.6.0",
        ],
        "check": (
            "import tpot, sklearn; "
            "from tpot import TPOTClassifier; "
            "import pkg_resources; "
            "print('TPOT:', tpot.__version__); "
            "print('Scikit-learn:', sklearn.__version__)"
        ),
    },
    "h2o": {
        "python": "3.11",
        "conda": ["pip", "openjdk=17"],
        "pip": [
            "numpy==1.26.4",
            "pandas==2.1.4",
            "scikit-learn==1.5.2",
            "matplotlib==3.8.4",
            "pyarrow==15.0.2",
            "joblib",
            "psutil",
            "h2o==3.46.0.11",
        ],
        "check": (
            "import h2o; "
            "from h2o.automl import H2OAutoML; "
            "print('H2O:', h2o.__version__)"
        ),
    },
    "autosklearn": {
        "python": "3.9",
        "conda": [
            "pip",
            "numpy=1.23.5",
            "pandas=1.5.3",
            "scipy=1.9.3",
            "scikit-learn=0.24.2",
            "pyarrow=12",
            "matplotlib=3.7.3",
            "joblib",
            "psutil",
            "auto-sklearn=0.15.0",
            "swig",
            "gcc_linux-64",
            "gxx_linux-64",
        ],
        "pip": [],
        "check": (
            "import autosklearn; "
            "import autosklearn.classification; "
            "print('Auto-sklearn:', autosklearn.__version__)"
        ),
    },
    "flaml": {
        "python": "3.11",
        "conda": ["pip"],
        "pip": [
            "numpy==1.26.4",
            "pandas==2.1.4",
            "scikit-learn==1.5.2",
            "matplotlib==3.8.4",
            "pyarrow==15.0.2",
            "joblib",
            "psutil",
            "flaml[automl]==2.6.0",
            "xgboost==2.1.4",
            "lightgbm==4.6.0",
        ],
        "check": (
            "import flaml; "
            "from flaml import AutoML; "
            "print('FLAML:', flaml.__version__)"
        ),
    },
    "lazypredict": {
        "python": "3.11",
        "conda": ["pip"],
        "pip": [
            "numpy==1.26.4",
            "pandas==2.1.4",
            "scikit-learn==1.5.2",
            "matplotlib==3.8.4",
            "pyarrow==15.0.2",
            "joblib",
            "psutil",
            "lazypredict==0.2.16",
            "xgboost==2.1.4",
            "lightgbm==4.6.0",
        ],
        "check": (
            "import lazypredict; "
            "from lazypredict.Supervised import LazyClassifier; "
            "print('LazyPredict:', lazypredict.__version__)"
        ),
    },
    "autopytorch": {
        "python": "3.8",
        "conda": [
            "pip",
            "swig=3.0.12",
            "numpy=1.23.5",
            "pandas=1.5.3",
            "scipy=1.9.3",
            "scikit-learn=0.24.2",
            "pyarrow=12",
            "matplotlib=3.7.3",
            "joblib",
            "psutil",
            "gcc_linux-64",
            "gxx_linux-64",
        ],
        # Ferramentas antigas são necessárias para compilar dependências
        # publicadas antes do PEP 517/518 atual.
        "pip_toolchain": [
            "pip==23.3.2",
            "setuptools==68.2.2",
            "wheel==0.41.3",
            "Cython==0.29.36",
        ],
        # A instalação é separada em etapas para evitar que o resolvedor tente
        # trocar a versão do PyTorch ou do ConfigSpace no mesmo comando.
        "pip_steps": [
            [
                "install",
                "--no-cache-dir",
                "torch==1.13.1+cu117",
                "torchvision==0.14.1+cu117",
                "--extra-index-url",
                "https://download.pytorch.org/whl/cu117",
            ],
            [
                "install",
                "--no-cache-dir",
                "--no-build-isolation",
                # Compatibilidade do Auto-PyTorch 0.2.1:
                # - Auto-PyTorch exige ConfigSpace >= 0.5.0;
                # - SMAC 1.2 exige ConfigSpace < 0.5 e causa conflito;
                # - SMAC 1.3.4 aceita ConfigSpace >= 0.5.0.
                "ConfigSpace==0.5.0",
                "smac==1.3.4",
                "pynisher==0.6.4",
                "pyrfr==0.8.3",
                "flaky==3.8.1",
            ],
            [
                "install",
                "--no-cache-dir",
                "--no-build-isolation",
                "autoPyTorch==0.2.1",
            ],
        ],
        "pip": [],
        "check": (
            "from importlib.metadata import version; "
            "import numpy, pandas, scipy, sklearn, pyarrow, matplotlib, joblib; "
            "import torch, torchvision, autoPyTorch, ConfigSpace, smac; "
            "from autoPyTorch.api.tabular_classification "
            "import TabularClassificationTask; "
            "from autoPyTorch.datasets.resampling_strategy "
            "import HoldoutValTypes; "
            "configspace_version = version('ConfigSpace'); "
            "smac_version = version('smac'); "
            "sklearn_version = version('scikit-learn'); "
            "assert configspace_version == '0.5.0', configspace_version; "
            "assert smac_version == '1.3.4', smac_version; "
            "assert sklearn_version.startswith('0.24.'), sklearn_version; "
            "print('ConfigSpace:', configspace_version); "
            "print('SMAC:', smac_version); "
            "print('Scikit-learn:', sklearn_version); "
            "print('PyTorch:', torch.__version__); "
            "print('Torchvision:', torchvision.__version__); "
            "print('CUDA:', torch.cuda.is_available()); "
            "print('Auto-PyTorch:', autoPyTorch.__version__); "
            "print('Auto-PyTorch validado com sucesso.')"
        ),
    },
    "autokeras": {
        "python": "3.11",
        "conda": ["pip"],
        "pip": [
            "numpy==1.26.4",
            "pandas==2.1.4",
            "scikit-learn==1.5.2",
            "matplotlib==3.8.4",
            "pyarrow==15.0.2",
            "joblib",
            "psutil",
            "tensorflow==2.20.0",
            "keras-tuner==1.4.7",
            "autokeras==3.0.0",
        ],
        "check": (
            "import tensorflow as tf, keras, autokeras; "
            "print('TensorFlow:', tf.__version__); "
            "print('Keras:', keras.__version__); "
            "print('AutoKeras:', autokeras.__version__); "
            "print('GPU:', bool(tf.config.list_physical_devices('GPU')))"
        ),
    },
    "autogluon": {
        "python": "3.11",
        "conda": ["pip"],
        "pip_toolchain": [
            "pip==25.1.1",
            "setuptools==80.9.0",
            "wheel==0.45.1",
        ],
        # Pilha numérica fixada para impedir combinações binárias instáveis
        # entre NumPy, SciPy, pandas, scikit-learn, PyArrow e Matplotlib.
        "pip": [
            "numpy==1.26.4",
            "scipy==1.15.3",
            "pandas==2.2.3",
            "scikit-learn==1.6.1",
            "pyarrow==19.0.1",
            "matplotlib==3.9.4",
            "joblib==1.4.2",
            "psutil==6.1.1",
            "typing_extensions==4.12.2",
            "autogluon.tabular[lightgbm,xgboost]==1.5.0",
        ],
        "check": (
            "import numpy, scipy, pandas, sklearn, pyarrow, matplotlib, joblib; "
            "import typing_extensions; "
            "import autogluon.tabular as agt; "
            "from autogluon.tabular import TabularPredictor; "
            "assert numpy.__version__ == '1.26.4', numpy.__version__; "
            "assert pandas.__version__ == '2.2.3', pandas.__version__; "
            "assert sklearn.__version__ == '1.6.1', sklearn.__version__; "
            "print('NumPy:', numpy.__version__); "
            "print('pandas:', pandas.__version__); "
            "print('Scikit-learn:', sklearn.__version__); "
            "print('PyArrow:', pyarrow.__version__); "
            "print('Matplotlib:', matplotlib.__version__); "
            "print('AutoGluon:', agt.__version__)"
        ),
    },
}

def executar_comando(
    cmd,
    *,
    log_path=None,
    env=None,
    check=True,
):
    """Executa o comando, mostra a saída e grava um log completo."""
    cmd = [str(x) for x in cmd]
    print("\n$", " ".join(cmd))
    print("-" * 100)

    # Cria uma cópia das variáveis de ambiente do Colab.
    ambiente = os.environ.copy()

    # Remove o backend gráfico herdado do Jupyter/Google Colab.
    ambiente.pop("MPLBACKEND", None)

    # Força um backend não interativo compatível com subprocessos.
    ambiente["MPLBACKEND"] = "Agg"

    # Adiciona outras variáveis de ambiente, quando fornecidas.
    if env:
        ambiente.update(env)

    processo = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=ambiente,
    )

    linhas = []
    if processo.stdout is not None:
        for linha in processo.stdout:
            print(linha, end="")
            linhas.append(linha)

    returncode = processo.wait()
    saida = "".join(linhas)

    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text(
            saida,
            encoding="utf-8",
            errors="replace",
        )

    resultado = subprocess.CompletedProcess(
        args=cmd,
        returncode=returncode,
        stdout=saida,
        stderr=None,
    )

    if check and returncode != 0:
        erro = subprocess.CalledProcessError(
            returncode,
            cmd,
            output=saida,
        )
        erro.log_path = str(log_path) if log_path is not None else None
        raise erro

    return resultado


def instalar_micromamba():
    if MICROMAMBA.exists():
        print(f"Micromamba já instalado: {MICROMAMBA}")
        return

    Path("/content/bin").mkdir(
        parents=True,
        exist_ok=True,
    )
    comando = (
        "curl -Ls "
        "https://micro.mamba.pm/api/micromamba/linux-64/latest "
        "| tar -xj -C /content bin/micromamba"
    )
    subprocess.run(
        comando,
        shell=True,
        check=True,
    )

    if not MICROMAMBA.exists():
        raise FileNotFoundError(
            "Falha ao instalar o Micromamba."
        )

    print(f"Micromamba instalado: {MICROMAMBA}")


def nome_ambiente(framework):
    return f"ciciot_{framework}"


from pathlib import Path


def caminho_ambiente(framework):
    """
    Retorna o caminho completo do ambiente Micromamba
    correspondente ao framework informado.
    """
    return (
        Path(MAMBA_ROOT_PREFIX)
        / "envs"
        / nome_ambiente(framework)
    )


def ambiente_existe(framework):
    return caminho_ambiente(framework).exists()


def remover_ambiente(framework):
    env_name = nome_ambiente(framework)

    if ambiente_existe(framework):
        executar_comando(
            [
                MICROMAMBA,
                "env",
                "remove",
                "-y",
                "-n",
                env_name,
            ],
            log_path=(
                LOGS_DIR
                / f"{framework}_remocao.log"
            ),
            check=False,
        )


def variaveis_ambiente_framework(framework):
    """Retorna variáveis específicas necessárias ao framework."""
    variaveis = {}

    if framework == "pycaret":
        env_lib = str(caminho_ambiente(framework) / "lib")
        ld_library_path_atual = os.environ.get(
            "LD_LIBRARY_PATH",
            "",
        )

        if ld_library_path_atual:
            variaveis["LD_LIBRARY_PATH"] = (
                env_lib
                + os.pathsep
                + ld_library_path_atual
            )
        else:
            variaveis["LD_LIBRARY_PATH"] = env_lib

    return variaveis


def verificar_ambiente(framework):
    spec = FRAMEWORK_ENV_SPECS[framework]
    env_name = nome_ambiente(framework)

    script = (
        "import sys; "
        "print('Python:', sys.version); "
        "print('Executável:', sys.executable); "
        + spec["check"]
    )

    comando = [
        MICROMAMBA,
        "run",
        "-n",
        env_name,
        "python",
        "-c",
        script,
    ]

    return executar_comando(
        comando,
        log_path=(
            LOGS_DIR
            / f"{framework}_verificacao.log"
        ),
        env=variaveis_ambiente_framework(framework),
        check=True,
    )


def criar_ambiente(framework):
    if framework not in FRAMEWORK_ENV_SPECS:
        raise KeyError(
            f"Especificação não encontrada: {framework}"
        )

    spec = FRAMEWORK_ENV_SPECS[framework]
    env_name = nome_ambiente(framework)

    # Ambientes válidos são reutilizados. Se a verificação de importação
    # falhar (inclusive por CXXABI no PyCaret), o ambiente será recriado abaixo.
    frameworks_recriar = set()
    if framework in frameworks_recriar and ambiente_existe(framework):
        print(
            f"Recriando o ambiente corrigido: {env_name}"
        )
        remover_ambiente(framework)

    elif ambiente_existe(framework) and REUTILIZAR_AMBIENTES:
        try:
            verificar_ambiente(framework)
            print(f"Reutilizando ambiente: {env_name}")
            return env_name
        except Exception:
            print(
                f"Ambiente {env_name} inválido. "
                "Ele será recriado."
            )
            remover_ambiente(framework)

    elif ambiente_existe(framework):
        remover_ambiente(framework)

    canais = ["-c", "conda-forge"]
    if framework == "autopytorch":
        canais += [
            "-c",
            "pytorch",
            "-c",
            "nvidia",
        ]

    criar_cmd = [
        MICROMAMBA,
        "create",
        "-y",
        "-n",
        env_name,
        *canais,
        f"python={spec['python']}",
        *spec["conda"],
    ]

    executar_comando(
        criar_cmd,
        log_path=(
            LOGS_DIR
            / f"{framework}_criacao.log"
        ),
        check=True,
    )

    # Não usar sempre a versão mais recente de pip/setuptools em ambientes
    # legados. Cada framework pode declarar sua própria toolchain.
    toolchain = spec.get(
        "pip_toolchain",
        ["pip", "wheel"],
    )
    executar_comando(
        [
            MICROMAMBA,
            "run",
            "-n",
            env_name,
            "python",
            "-m",
            "pip",
            "install",
            "--upgrade",
            *toolchain,
        ],
        log_path=(
            LOGS_DIR
            / f"{framework}_pip_upgrade.log"
        ),
        env=variaveis_ambiente_framework(framework),
        check=True,
    )

    # Etapas especiais, usadas principalmente pelo Auto-PyTorch.
    for numero, argumentos in enumerate(
        spec.get("pip_steps", []),
        start=1,
    ):
        executar_comando(
            [
                MICROMAMBA,
                "run",
                "-n",
                env_name,
                "python",
                "-m",
                "pip",
                *argumentos,
            ],
            log_path=(
                LOGS_DIR
                / f"{framework}_instalacao_etapa_{numero}.log"
            ),
            env=variaveis_ambiente_framework(framework),
            check=True,
        )

    pip_pkgs = spec.get("pip", [])
    if pip_pkgs:
        executar_comando(
            [
                MICROMAMBA,
                "run",
                "-n",
                env_name,
                "python",
                "-m",
                "pip",
                "install",
                "--no-cache-dir",
                *pip_pkgs,
            ],
            log_path=(
                LOGS_DIR
                / f"{framework}_instalacao.log"
            ),
            env=variaveis_ambiente_framework(framework),
            check=True,
        )

    verificar_ambiente(framework)
    return env_name


def executar_no_ambiente(framework, config_path):
    env_name = nome_ambiente(framework)
    env_execucao = variaveis_ambiente_framework(framework)
    env_execucao.update(
        {
            "PYTHONUNBUFFERED": "1",
            "MPLBACKEND": "Agg",
            "OMP_NUM_THREADS": str(max(1, int(N_JOBS))),
            "MKL_NUM_THREADS": str(max(1, int(N_JOBS))),
        }
    )

    return executar_comando(
        [
            MICROMAMBA,
            "run",
            "-n",
            env_name,
            "python",
            str(RUNNER_SCRIPT),
            "--framework",
            framework,
            "--config",
            str(config_path),
        ],
        log_path=(
            LOGS_DIR
            / f"{framework}_execucao.log"
        ),
        env=env_execucao,
        check=True,
    )

In [11]:
# ============================================================
# TESTE DO AMBIENTE PYCARET
# ============================================================

from datetime import datetime
from zoneinfo import ZoneInfo

# Define minimal required variables for testing environment creation
RESULTS_DIR = Path("/content/drive/MyDrive/CICIoT2023_Resultados/AutoML_Comparativo_Unificado")
RESULTS_DIR.mkdir(parents=True, exist_ok=True) # Ensure it exists
FUSO_HORARIO = ZoneInfo("America/Sao_Paulo")
RUN_ID_GERAL = datetime.now(FUSO_HORARIO).strftime("%Y%m%d_%H%M%S")
# For a specific framework test, simulate the scenario-specific directories
CENARIO_TEST = "pycaret_test" # A distinct name for this test run
RUN_RESULTS_DIR = RESULTS_DIR / RUN_ID_GERAL / CENARIO_TEST
LOGS_DIR = RUN_RESULTS_DIR / "_logs"
LOGS_DIR.mkdir(parents=True, exist_ok=True) # Ensure logs directory exists

print("=" * 100)
print("ETAPA 1 — INSTALAÇÃO OU LOCALIZAÇÃO DO MICROMAMBA")
print("=" * 100)

instalar_micromamba()


print("\n" + "=" * 100)
print("ETAPA 2 — CRIAÇÃO OU CORREÇÃO DO AMBIENTE PYCARET")
print("=" * 100)

criar_ambiente("pycaret")


print("\n" + "=" * 100)
print("ETAPA 3 — VERIFICAÇÃO FINAL DO AMBIENTE PYCARET")
print("=" * 100)

verificar_ambiente("pycaret")


print("\n" + "=" * 100)
print("AMBIENTE PYCARET PREPARADO COM SUCESSO")
print("=" * 100)

ETAPA 1 — INSTALAÇÃO OU LOCALIZAÇÃO DO MICROMAMBA
Micromamba já instalado: /content/bin/micromamba

ETAPA 2 — CRIAÇÃO OU CORREÇÃO DO AMBIENTE PYCARET

$ /content/bin/micromamba run -n ciciot_pycaret python -c import sys; print('Python:', sys.version); print('Executável:', sys.executable); import pycaret; import sklearn; import xgboost; from pycaret.classification import setup; print('PyCaret:', pycaret.__version__); print('Scikit-learn:', sklearn.__version__); print('XGBoost:', xgboost.__version__)
----------------------------------------------------------------------------------------------------
Python: 3.10.20 | packaged by conda-forge | (main, Jun 11 2026, 03:31:56) [GCC 14.3.0]
Executável: /content/micromamba/envs/ciciot_pycaret/bin/python
PyCaret: 3.3.2
Scikit-learn: 1.4.2
XGBoost: 2.1.4
Reutilizando ambiente: ciciot_pycaret

ETAPA 3 — VERIFICAÇÃO FINAL DO AMBIENTE PYCARET

$ /content/bin/micromamba run -n ciciot_pycaret python -c import sys; print('Python:', sys.version); print(

In [12]:
# ============================================================
# TESTE DO XGBOOST NO CATÁLOGO INTERNO DO PYCARET
# ============================================================

import os
import subprocess
import textwrap
from pathlib import Path

MICROMAMBA = "/content/bin/micromamba"
MAMBA_ROOT_PREFIX = "/content/micromamba"
ENV_NAME = "ciciot_pycaret"

codigo_teste = textwrap.dedent(
    """
    import pandas as pd

    from pycaret.classification import (
        models,
        setup,
    )

    dados = pd.DataFrame(
        {
            "feature_1": list(range(40)),
            "feature_2": [0, 1] * 20,
            "classe": [
                "Benign",
                "Malicious",
            ] * 20,
        }
    )

    setup(
        data=dados,
        target="classe",
        session_id=42,
        train_size=0.80,
        fold=2,
        fold_shuffle=True,
        data_split_stratify=True,
        n_jobs=1,
        html=False,
        verbose=False,
    )

    catalogo = models(
        internal=True
    )

    disponivel = (
        "xgboost" in catalogo.index
    )

    print("=" * 100)
    print("VERIFICAÇÃO DO CATÁLOGO DO PYCARET")
    print("=" * 100)

    print(
        "XGBoost disponível no catálogo:",
        disponivel,
    )

    if disponivel:
        print()
        print(
            catalogo.loc[
                ["xgboost"]
            ].to_string()
        )
    else:
        print()
        print("Modelos disponíveis:")
        print(
            sorted(
                catalogo.index
                .astype(str)
                .tolist()
            )
        )

        raise RuntimeError(
            "O XGBoost foi importado, mas não foi "
            "registrado no catálogo do PyCaret."
        )

    print()
    print("=" * 100)
    print("TESTE CONCLUÍDO COM SUCESSO")
    print("=" * 100)
    """
)

env_execucao = os.environ.copy()
env_execucao["MAMBA_ROOT_PREFIX"] = (
    MAMBA_ROOT_PREFIX
)

# Add LD_LIBRARY_PATH to prioritize the micromamba environment's libraries
pycaret_env_lib = Path(MAMBA_ROOT_PREFIX) / "envs" / ENV_NAME / "lib"
current_ld_library_path = env_execucao.get("LD_LIBRARY_PATH", "")
if current_ld_library_path:
    env_execucao["LD_LIBRARY_PATH"] = str(pycaret_env_lib) + os.pathsep + current_ld_library_path
else:
    env_execucao["LD_LIBRARY_PATH"] = str(pycaret_env_lib)

cmd = [
    MICROMAMBA,
    "run",
    "-n",
    ENV_NAME,
    "python",
    "-c",
    codigo_teste,
]

resultado = subprocess.run(
    cmd,
    check=False,
    env=env_execucao,
    text=True,
    capture_output=True,
)

print(resultado.stdout)

if resultado.stderr:
    print("STDERR:")
    print(resultado.stderr)

if resultado.returncode != 0:
    raise RuntimeError(
        "O teste do catálogo do PyCaret falhou."
    )


VERIFICAÇÃO DO CATÁLOGO DO PYCARET
XGBoost disponível no catálogo: True

                              Name                      Reference  Turbo  Special                                    Class                                                                Equality                                                                                                            Args                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

## 8. Executor compartilhado dos dez frameworks

In [13]:
#================================================================
# 7. Executor compartilhado corrigido
#===============================================================


RUNNER_CODE = r'''
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Executor isolado de frameworks AutoML para o CICIoT2023.
Este arquivo é gerado automaticamente pelo notebook orquestrador.
"""

from __future__ import annotations

import argparse
import gc
import json
import os
import pickle
import shutil
import sys
import time
import traceback
import warnings
from pathlib import Path
from typing import Any, Optional

warnings.filterwarnings("ignore")

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler


def carregar_config(caminho: str | Path) -> dict[str, Any]:
    with open(caminho, "r", encoding="utf-8") as f:
        return json.load(f)


def garantir_diretorio(caminho: Path) -> None:
    caminho.mkdir(parents=True, exist_ok=True)


def limpar_diretorio_se_solicitado(caminho: Path, apagar: bool) -> None:
    if apagar and caminho.exists():
        shutil.rmtree(caminho)
    caminho.mkdir(parents=True, exist_ok=True)


def salvar_json(obj: Any, caminho: Path) -> None:
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)


def salvar_dataframe_png(
    df: pd.DataFrame,
    caminho: Path,
    titulo: str,
    max_linhas: int = 30,
    max_colunas: int = 14,
) -> None:
    tabela = df.head(max_linhas).iloc[:, :max_colunas].copy()
    largura = max(10, min(24, 1.4 * max(1, tabela.shape[1])))
    altura = max(3.5, min(18, 0.42 * (len(tabela) + 3)))
    fig, ax = plt.subplots(figsize=(largura, altura))
    ax.axis("off")
    tab = ax.table(
        cellText=tabela.astype(str).values,
        colLabels=[str(c) for c in tabela.columns],
        cellLoc="center",
        loc="center",
    )
    tab.auto_set_font_size(False)
    tab.set_fontsize(7)
    tab.scale(1, 1.25)
    ax.set_title(titulo, fontsize=12, pad=12)
    fig.tight_layout()
    fig.savefig(caminho, dpi=220, bbox_inches="tight")
    plt.close(fig)


def salvar_matriz_png(
    matriz: np.ndarray,
    labels: list[str],
    caminho: Path,
    titulo: str,
    formato: str = "d",
) -> None:
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(matriz)
    ax.set_title(titulo)
    ax.set_xlabel("Classe predita")
    ax.set_ylabel("Classe real")
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_yticklabels(labels)

    limite = float(np.nanmax(matriz)) / 2 if matriz.size else 0.0
    for i in range(matriz.shape[0]):
        for j in range(matriz.shape[1]):
            valor = matriz[i, j]
            texto = format(valor, formato)
            ax.text(
                j,
                i,
                texto,
                ha="center",
                va="center",
                color="white" if float(valor) > limite else "black",
            )

    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(caminho, dpi=250, bbox_inches="tight")
    plt.close(fig)


def preparar_dados_numericos(
    train_path: Path,
    test_path: Path,
    label_col: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series, dict[str, Any]]:
    """
    Carrega o split compartilhado e aplica o mesmo pré-processamento numérico
    a todos os frameworks para tornar a comparação mais justa.
    """
    train_df = pd.read_parquet(train_path)
    test_df = pd.read_parquet(test_path)

    if label_col not in train_df.columns or label_col not in test_df.columns:
        raise KeyError(f"Coluna alvo ausente: {label_col}")

    y_train = train_df.pop(label_col).astype(str).reset_index(drop=True)
    y_test = test_df.pop(label_col).astype(str).reset_index(drop=True)

    colunas_descartar = [
        c for c in train_df.columns
        if c.startswith("__") or c.lower() in {
            "source_file", "filename", "label_original", "label", "class",
            "attack", "category", "type"
        }
    ]
    train_df.drop(columns=colunas_descartar, errors="ignore", inplace=True)
    test_df.drop(columns=colunas_descartar, errors="ignore", inplace=True)

    # Mantém exatamente as mesmas colunas e a mesma ordem.
    colunas = list(train_df.columns)
    faltantes_teste = [c for c in colunas if c not in test_df.columns]
    for c in faltantes_teste:
        test_df[c] = np.nan
    test_df = test_df.reindex(columns=colunas)

    # Os atributos do CICIoT2023 são numéricos. Qualquer valor textual
    # inesperado é convertido para NaN e depois imputado pela mediana do treino.
    for c in colunas:
        train_df[c] = pd.to_numeric(train_df[c], errors="coerce")
        test_df[c] = pd.to_numeric(test_df[c], errors="coerce")

    train_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    test_df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # Remove colunas sem qualquer valor válido no treino.
    colunas_totalmente_vazias = [c for c in colunas if train_df[c].notna().sum() == 0]
    if colunas_totalmente_vazias:
        train_df.drop(columns=colunas_totalmente_vazias, inplace=True)
        test_df.drop(columns=colunas_totalmente_vazias, inplace=True)

    # Remove colunas constantes no treino.
    colunas_constantes = [
        c for c in train_df.columns
        if train_df[c].nunique(dropna=True) <= 1
    ]
    if colunas_constantes:
        train_df.drop(columns=colunas_constantes, inplace=True)
        test_df.drop(columns=colunas_constantes, inplace=True)

    imputer = SimpleImputer(strategy="median")
    x_train_np = imputer.fit_transform(train_df)
    x_test_np = imputer.transform(test_df)

    X_train = pd.DataFrame(
        x_train_np.astype(np.float32, copy=False),
        columns=train_df.columns,
    )
    X_test = pd.DataFrame(
        x_test_np.astype(np.float32, copy=False),
        columns=train_df.columns,
    )

    info = {
        "train_shape": list(X_train.shape),
        "test_shape": list(X_test.shape),
        "classes_train": y_train.value_counts().to_dict(),
        "classes_test": y_test.value_counts().to_dict(),
        "colunas_removidas_vazias": colunas_totalmente_vazias,
        "colunas_removidas_constantes": colunas_constantes,
        "numero_features": int(X_train.shape[1]),
    }

    del train_df, test_df, x_train_np, x_test_np
    gc.collect()
    return X_train, X_test, y_train, y_test, info

def obter_score_positivo(
    y_proba: Any,
    classes: Optional[list[str]],
    classe_positiva: Optional[str],
) -> Optional[np.ndarray]:
    if y_proba is None or classe_positiva is None:
        return None

    if isinstance(y_proba, pd.DataFrame):
        if classe_positiva in y_proba.columns:
            return y_proba[classe_positiva].to_numpy(dtype=float)
        if y_proba.shape[1] == 2:
            return y_proba.iloc[:, 1].to_numpy(dtype=float)

    arr = np.asarray(y_proba)
    if arr.ndim == 1:
        return arr.astype(float)
    if arr.ndim == 2 and arr.shape[1] >= 2:
        if classes and classe_positiva in classes:
            return arr[:, classes.index(classe_positiva)].astype(float)
        return arr[:, 1].astype(float)
    if arr.ndim == 2 and arr.shape[1] == 1:
        return arr[:, 0].astype(float)
    return None


def normalizar_predicoes(
    y_pred: Any,
    classes: list[str],
    label_encoder: Optional[LabelEncoder] = None,
) -> np.ndarray:
    arr = np.asarray(y_pred).reshape(-1)

    if label_encoder is not None and np.issubdtype(arr.dtype, np.number):
        arr_int = np.rint(arr).astype(int)
        return label_encoder.inverse_transform(arr_int)

    # Alguns frameworks devolvem 0/1 mesmo sem LabelEncoder explícito.
    if np.issubdtype(arr.dtype, np.number):
        unicos = set(np.unique(arr).tolist())
        if unicos.issubset(set(range(len(classes)))):
            return np.array([classes[int(round(v))] for v in arr], dtype=object)

    return arr.astype(str)


def avaliar_e_salvar(
    framework: str,
    y_test: pd.Series,
    y_pred: Any,
    y_proba: Any,
    result_dir: Path,
    classes: list[str],
    classe_positiva: Optional[str],
    leaderboard: Optional[pd.DataFrame] = None,
    extra_info: Optional[dict[str, Any]] = None,
    label_encoder: Optional[LabelEncoder] = None,
) -> dict[str, float]:
    garantir_diretorio(result_dir)
    y_true = np.asarray(y_test.astype(str))
    y_pred_norm = normalizar_predicoes(y_pred, classes, label_encoder=label_encoder)
    if len(y_true) != len(y_pred_norm):
        raise ValueError(
            f"Quantidade de predições ({len(y_pred_norm)}) diferente do teste ({len(y_true)})."
        )

    score = obter_score_positivo(
        y_proba,
        list(label_encoder.classes_) if label_encoder is not None else classes,
        classe_positiva,
    )

    matriz = confusion_matrix(y_true, y_pred_norm, labels=classes)
    matriz_norm = confusion_matrix(y_true, y_pred_norm, labels=classes, normalize="true")

    metricas: dict[str, float] = {
        "Accuracy": float(accuracy_score(y_true, y_pred_norm)),
        "Balanced Accuracy": float(balanced_accuracy_score(y_true, y_pred_norm)),
        "Precision Macro": float(precision_score(y_true, y_pred_norm, average="macro", zero_division=0)),
        "Recall Macro": float(recall_score(y_true, y_pred_norm, average="macro", zero_division=0)),
        "Macro-F1": float(f1_score(y_true, y_pred_norm, average="macro", zero_division=0)),
        "Weighted-F1": float(f1_score(y_true, y_pred_norm, average="weighted", zero_division=0)),
        "MCC": float(matthews_corrcoef(y_true, y_pred_norm)),
    }

    if (
        classe_positiva is not None
        and len(classes) == 2
        and classe_positiva in classes
    ):
        metricas["Precision Malicious"] = float(
            precision_score(
                y_true, y_pred_norm,
                pos_label=classe_positiva,
                average="binary",
                zero_division=0,
            )
        )
        metricas["Recall Malicious"] = float(
            recall_score(
                y_true, y_pred_norm,
                pos_label=classe_positiva,
                average="binary",
                zero_division=0,
            )
        )
        metricas["F1 Malicious"] = float(
            f1_score(
                y_true, y_pred_norm,
                pos_label=classe_positiva,
                average="binary",
                zero_division=0,
            )
        )

    if score is not None and len(classes) == 2:
        y_bin = (y_true == classe_positiva).astype(int)
        try:
            metricas["ROC-AUC"] = float(roc_auc_score(y_bin, score))
        except Exception:
            metricas["ROC-AUC"] = np.nan
        try:
            metricas["Average Precision"] = float(average_precision_score(y_bin, score))
        except Exception:
            metricas["Average Precision"] = np.nan
    else:
        metricas["ROC-AUC"] = np.nan
        metricas["Average Precision"] = np.nan

    metricas_df = pd.DataFrame(
        {"Framework": framework, "Métrica": list(metricas), "Valor": list(metricas.values())}
    )
    metricas_df.to_csv(result_dir / f"metricas_{framework}.csv", index=False)
    salvar_dataframe_png(
        metricas_df.round(6),
        result_dir / f"metricas_{framework}.png",
        f"Métricas — {framework}",
    )

    report = classification_report(
        y_true,
        y_pred_norm,
        labels=classes,
        target_names=classes,
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(result_dir / f"classification_report_{framework}.csv")
    salvar_dataframe_png(
        report_df.round(6).reset_index(),
        result_dir / f"classification_report_{framework}.png",
        f"Classification report — {framework}",
    )

    cm_df = pd.DataFrame(
        matriz,
        index=[f"Real_{c}" for c in classes],
        columns=[f"Pred_{c}" for c in classes],
    )
    cm_df.to_csv(result_dir / f"matriz_confusao_{framework}.csv")
    salvar_matriz_png(
        matriz,
        classes,
        result_dir / f"matriz_confusao_{framework}.png",
        f"Matriz de confusão absoluta — {framework}",
        formato="d",
    )

    cm_norm_df = pd.DataFrame(
        matriz_norm,
        index=[f"Real_{c}" for c in classes],
        columns=[f"Pred_{c}" for c in classes],
    )
    cm_norm_df.to_csv(result_dir / f"matriz_confusao_normalizada_{framework}.csv")
    salvar_matriz_png(
        matriz_norm,
        classes,
        result_dir / f"matriz_confusao_normalizada_{framework}.png",
        f"Matriz de confusão normalizada — {framework}",
        formato=".4f",
    )

    pred_df = pd.DataFrame({"y_real": y_true, "y_pred": y_pred_norm})
    if score is not None and len(score) == len(pred_df):
        pred_df["score_malicious"] = score
    pred_df.to_csv(result_dir / f"inferencias_{framework}.csv", index=False)

    if leaderboard is not None and len(leaderboard) > 0:
        leaderboard = pd.DataFrame(leaderboard)
        leaderboard.to_csv(result_dir / f"leaderboard_{framework}.csv", index=False)
        salvar_dataframe_png(
            leaderboard,
            result_dir / f"leaderboard_{framework}.png",
            f"Leaderboard — {framework}",
        )

    if extra_info:
        salvar_json(extra_info, result_dir / f"info_{framework}.json")

    return metricas


def executar_pycaret(X_train, X_test, y_train, config, result_dir):
    from pycaret.classification import (
        add_metric,
        compare_models,
        finalize_model,
        load_model,
        predict_model,
        models,
        pull,
        save_model,
        setup,
    )

    label_col = config["LABEL_FINAL"]
    model_base = result_dir / "modelo_pycaret"
    model_file = Path(str(model_base) + ".pkl")
    data = X_train.copy()
    data[label_col] = y_train.values

    setup(
        data=data,
        target=label_col,
        session_id=config["RANDOM_STATE"],
        train_size=1.0 - config["VALIDATION_SPLIT"],
        data_split_stratify=config["STRATIFY"],
        fold=config.get("PYCARET_FOLDS", 5),
        fold_shuffle=True,
        n_jobs=config["N_JOBS"],
        html=False,
        verbose=False,
    )
    # ========================================================
    # VALIDAÇÃO DOS MODELOS DISPONÍVEIS NO PYCARET
    # ========================================================

    catalogo_modelos = models(
        internal=True
    )

    ids_disponiveis = set(
        catalogo_modelos.index
        .astype(str)
        .tolist()
    )

    modelos_solicitados = config.get(
        "PYCARET_INCLUDE_MODELS",
        [],
    )

    modelos_solicitados = [
        str(modelo).strip()
        for modelo in modelos_solicitados
        if str(modelo).strip()
    ]

    modelos_indisponiveis = [
        modelo
        for modelo in modelos_solicitados
        if modelo not in ids_disponiveis
    ]

    print("=" * 100)
    print("VALIDAÇÃO DOS MODELOS DO PYCARET")
    print("=" * 100)
    print(
        "Modelos solicitados:",
        modelos_solicitados,
    )
    print(
        "Modelos indisponíveis:",
        modelos_indisponiveis,
    )

    if modelos_indisponiveis:
        raise RuntimeError(
            "Os seguintes modelos solicitados não estão "
            "disponíveis no catálogo do PyCaret: "
            f"{modelos_indisponiveis}. "
            "Verifique as dependências instaladas no ambiente "
            "ciciot_pycaret."
        )

    print(
        "Todos os modelos solicitados estão disponíveis."
    )
    print("=" * 100)
    try:
        add_metric(
            id="macro_f1",
            name="Macro F1",
            score_func=f1_score,
            greater_is_better=True,
            multiclass=True,
            average="macro",
            zero_division=0,
        )
    except Exception:
        pass

    if config["CARREGAR_MODELO_EXISTENTE"] and model_file.exists():
        modelo = load_model(str(model_base))
        leaderboard = None
    else:
        if not config["TREINAR_SE_NAO_EXISTIR"]:
            raise FileNotFoundError(model_file)

        kwargs_compare = {
            "sort": "Macro F1",
            "n_select": 1,
            "budget_time": max(1, int(config["TIME_LIMIT"] / 60)),
            "turbo": True,
            "verbose": True,
        }
        modelos_incluir = config.get("PYCARET_INCLUDE_MODELS")
        if modelos_incluir:
            kwargs_compare["include"] = modelos_incluir

        melhor = compare_models(**kwargs_compare)
        if melhor is None:
            raise RuntimeError("PyCaret não retornou um modelo válido.")

        leaderboard = pull().reset_index(drop=True)
        modelo = finalize_model(melhor)
        save_model(modelo, str(model_base))

    pred = predict_model(modelo, data=X_test, raw_score=True, verbose=False)
    pred_col = next(
        (c for c in ["prediction_label", "Label"] if c in pred.columns),
        None,
    )
    if pred_col is None:
        raise KeyError(
            f"Coluna de predição do PyCaret não encontrada. Colunas: {list(pred.columns)}"
        )
    y_pred = pred[pred_col].astype(str)

    score_col = next(
        (
            c for c in pred.columns
            if str(c).lower() in {
                "prediction_score_malicious",
                "score_malicious",
                "malicious",
            }
        ),
        None,
    )
    if score_col is None:
        score_col = next(
            (
                c for c in pred.columns
                if "prediction_score" in str(c).lower()
                and config["CLASSE_MALICIOSA"].lower() in str(c).lower()
            ),
            None,
        )
    y_proba = pred[score_col].to_numpy(dtype=float) if score_col else None

    info = {
        "search_metric": "Macro F1 customizado",
        "prediction_column": pred_col,
        "score_column": score_col,
    }
    return y_pred, y_proba, leaderboard, info


def executar_tpot(X_train, X_test, y_train, config, result_dir):
    from tpot import TPOTClassifier

    model_path = result_dir / "modelo_tpot.pkl"
    leaderboard = None

    if config["CARREGAR_MODELO_EXISTENTE"] and model_path.exists():
        modelo = joblib.load(model_path)
    else:
        if not config["TREINAR_SE_NAO_EXISTIR"]:
            raise FileNotFoundError(model_path)

        max_total_min = max(1, int(config["TIME_LIMIT"] / 60))
        max_eval_min = max(
            1,
            min(
                int(config.get("TPOT_MAX_EVAL_TIME_MINS", 2)),
                max(1, max_total_min // 3),
            ),
        )

        automl = TPOTClassifier(
            generations=config.get("TPOT_GENERATIONS", 2),
            population_size=config.get("TPOT_POPULATION_SIZE", 5),
            offspring_size=config.get("TPOT_POPULATION_SIZE", 5),
            scoring="f1_macro",
            cv=config.get("TPOT_CV", 3),
            random_state=config["RANDOM_STATE"],
            n_jobs=config["N_JOBS"],
            verbosity=2,
            max_time_mins=max_total_min,
            max_eval_time_mins=max_eval_min,
            disable_update_check=True,
            early_stop=config.get("TPOT_EARLY_STOP", 3),
        )
        automl.fit(X_train, y_train)

        modelo = getattr(automl, "fitted_pipeline_", None)
        if modelo is None:
            raise RuntimeError("TPOT terminou sem fitted_pipeline_.")

        joblib.dump(modelo, model_path)
        try:
            automl.export(str(result_dir / "pipeline_tpot_exportado.py"))
        except Exception as exc:
            (result_dir / "aviso_exportacao_tpot.txt").write_text(
                repr(exc), encoding="utf-8"
            )

        avaliados = getattr(automl, "evaluated_individuals_", None)
        if avaliados:
            leaderboard = (
                pd.DataFrame(avaliados)
                .transpose()
                .reset_index()
                .rename(columns={"index": "pipeline"})
            )

    y_pred = modelo.predict(X_test)
    y_proba = (
        modelo.predict_proba(X_test)
        if hasattr(modelo, "predict_proba")
        else None
    )
    return y_pred, y_proba, leaderboard, {"search_metric": "f1_macro"}


def executar_h2o(X_train, X_test, y_train, config, result_dir):
    import h2o
    from h2o.automl import H2OAutoML

    h2o.init(
        nthreads=int(config["H2O_NTHREADS"]),
        max_mem_size=str(config["H2O_MAX_MEM_SIZE"]),
    )
    try:
        h2o.no_progress()
    except Exception:
        pass

    label_col = config["LABEL_FINAL"]
    train_df = X_train.copy()
    train_df[label_col] = y_train.values
    test_features = X_test.copy()

    train_h2o = h2o.H2OFrame(train_df)
    test_h2o = h2o.H2OFrame(test_features)
    train_h2o[label_col] = train_h2o[label_col].asfactor()

    path_txt = result_dir / "h2o_model_path.txt"
    leaderboard = None

    if config["CARREGAR_MODELO_EXISTENTE"] and path_txt.exists():
        model_path = path_txt.read_text(encoding="utf-8").strip()
        if not model_path:
            raise RuntimeError("h2o_model_path.txt está vazio.")
        modelo = h2o.load_model(model_path)
    else:
        if not config["TREINAR_SE_NAO_EXISTIR"]:
            raise FileNotFoundError(path_txt)

        per_model = max(
            30,
            min(
                int(config["H2O_MAX_RUNTIME_PER_MODEL"]),
                int(config["TIME_LIMIT"]),
            ),
        )

        aml = H2OAutoML(
            max_runtime_secs=int(config["TIME_LIMIT"]),
            max_runtime_secs_per_model=per_model,
            max_models=int(config["H2O_MAX_MODELS"]),
            nfolds=int(config["H2O_NFOLDS"]),
            stopping_rounds=int(config["H2O_STOPPING_ROUNDS"]),
            stopping_tolerance=float(config["H2O_STOPPING_TOLERANCE"]),
            stopping_metric="mean_per_class_error",
            sort_metric="mean_per_class_error",
            balance_classes=False,
            seed=int(config["RANDOM_STATE"]),
            keep_cross_validation_models=bool(config["H2O_KEEP_CV_MODELS"]),
            keep_cross_validation_predictions=bool(config["H2O_KEEP_CV_PREDICTIONS"]),
            keep_cross_validation_fold_assignment=bool(
                config["H2O_KEEP_CV_FOLD_ASSIGNMENT"]
            ),
            verbosity="info",
        )
        aml.train(
            x=list(X_train.columns),
            y=label_col,
            training_frame=train_h2o,
        )

        modelo = aml.leader
        if modelo is None:
            raise RuntimeError("H2O AutoML terminou sem modelo líder.")

        model_path = h2o.save_model(
            modelo,
            path=str(result_dir),
            force=True,
        )
        path_txt.write_text(model_path, encoding="utf-8")
        leaderboard = aml.leaderboard.as_data_frame()

    pred_df = modelo.predict(test_h2o).as_data_frame()
    if "predict" not in pred_df.columns:
        raise KeyError(
            f"Coluna predict ausente na saída H2O: {list(pred_df.columns)}"
        )

    y_pred = pred_df["predict"].astype(str)
    y_proba = (
        pred_df
        if config["CLASSE_MALICIOSA"] in pred_df.columns
        else None
    )

    info = {
        "search_metric": "mean_per_class_error",
        "comparison_metric": config["EVAL_METRIC"],
        "h2o_model_id": getattr(modelo, "model_id", None),
    }
    return y_pred, y_proba, leaderboard, info


def executar_autosklearn(X_train, X_test, y_train, config, result_dir):
    import autosklearn.classification
    import autosklearn.metrics

    model_path = result_dir / "modelo_autosklearn.pkl"
    leaderboard = None
    tmp_dir = Path("/content") / f"autosklearn_tmp_{os.getpid()}"
    shutil.rmtree(tmp_dir, ignore_errors=True)

    if config["CARREGAR_MODELO_EXISTENTE"] and model_path.exists():
        modelo = joblib.load(model_path)
    else:
        if not config["TREINAR_SE_NAO_EXISTIR"]:
            raise FileNotFoundError(model_path)

        per_run = max(
            30,
            min(
                int(config.get("AUTOSKLEARN_PER_RUN_SECS", 180)),
                max(30, int(config["TIME_LIMIT"]) // 3),
            ),
        )

        modelo = autosklearn.classification.AutoSklearnClassifier(
            time_left_for_this_task=int(config["TIME_LIMIT"]),
            per_run_time_limit=per_run,
            metric=autosklearn.metrics.f1_macro,
            memory_limit=int(config["AUTOSKLEARN_MEMORY_MB"]),
            n_jobs=int(config["N_JOBS"]),
            seed=int(config["RANDOM_STATE"]),
            initial_configurations_via_metalearning=0,
            ensemble_kwargs={
                "ensemble_size": int(
                    config.get("AUTOSKLEARN_ENSEMBLE_SIZE", 10)
                )
            },
            ensemble_nbest=int(
                config.get("AUTOSKLEARN_ENSEMBLE_NBEST", 10)
            ),
            max_models_on_disc=int(
                config.get("AUTOSKLEARN_MAX_MODELS_ON_DISC", 10)
            ),
            resampling_strategy="holdout",
            resampling_strategy_arguments={
                "train_size": 1.0 - float(config["VALIDATION_SPLIT"]),
                "shuffle": True,
            },
            tmp_folder=str(tmp_dir),
            delete_tmp_folder_after_terminate=False,
        )
        modelo.fit(X_train, y_train)
        joblib.dump(modelo, model_path)

    y_pred = modelo.predict(X_test)
    y_proba = (
        modelo.predict_proba(X_test)
        if hasattr(modelo, "predict_proba")
        else None
    )

    try:
        leaderboard = modelo.leaderboard()
    except Exception:
        leaderboard = pd.DataFrame(
            [{"statistics": getattr(modelo, "sprint_statistics", lambda: "")()}]
        )

    info = {
        "search_metric": "autosklearn.metrics.f1_macro",
        "temporary_directory": str(tmp_dir),
    }
    return y_pred, y_proba, leaderboard, info



def executar_sklearn(X_train, X_test, y_train, config, result_dir):
    """Seleciona o melhor ensemble clássico por Macro-F1 estratificado."""
    from sklearn.ensemble import (
        ExtraTreesClassifier,
        HistGradientBoostingClassifier,
        RandomForestClassifier,
    )
    from sklearn.model_selection import StratifiedKFold, cross_val_score

    seed = int(config["RANDOM_STATE"])
    n_jobs = int(config["N_JOBS"])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    candidatos = {
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=500, class_weight="balanced",
            max_features="sqrt", min_samples_leaf=1,
            n_jobs=n_jobs, random_state=seed,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=500, class_weight="balanced_subsample",
            max_features="sqrt", min_samples_leaf=1,
            n_jobs=n_jobs, random_state=seed,
        ),
        "HistGradientBoosting": HistGradientBoostingClassifier(
            learning_rate=0.05, max_iter=500, max_leaf_nodes=31,
            l2_regularization=1.0, early_stopping=True,
            random_state=seed,
        ),
    }
    linhas = []
    melhor_nome, melhor_score = None, -np.inf
    for nome, modelo in candidatos.items():
        scores = cross_val_score(
            modelo, X_train, y_train, scoring="f1_macro",
            cv=cv, n_jobs=n_jobs, error_score="raise",
        )
        media = float(np.mean(scores))
        linhas.append({
            "modelo": nome, "Macro-F1 CV": media,
            "Desvio CV": float(np.std(scores)),
        })
        if media > melhor_score:
            melhor_nome, melhor_score = nome, media

    modelo = candidatos[melhor_nome]
    modelo.fit(X_train, y_train)
    joblib.dump(modelo, result_dir / "modelo_sklearn.joblib")
    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test) if hasattr(modelo, "predict_proba") else None
    leaderboard = pd.DataFrame(linhas).sort_values("Macro-F1 CV", ascending=False)
    info = {
        "melhor_modelo": melhor_nome,
        "macro_f1_cv": melhor_score,
        "search_metric": "f1_macro",
    }
    return y_pred, y_proba, leaderboard, info


def executar_flaml(X_train, X_test, y_train, config, result_dir):
    from flaml import AutoML

    model_path = result_dir / "modelo_flaml.pkl"

    if config["CARREGAR_MODELO_EXISTENTE"] and model_path.exists():
        try:
            modelo = AutoML.load_pickle(str(model_path))
        except Exception:
            modelo = joblib.load(model_path)
    else:
        if not config["TREINAR_SE_NAO_EXISTIR"]:
            raise FileNotFoundError(model_path)

        modelo = AutoML()
        fit_kwargs = {
            "X_train": X_train,
            "y_train": y_train,
            "task": "classification",
            "time_budget": int(config["TIME_LIMIT"]),
            "metric": "macro_f1",
            "eval_method": "holdout",
            "split_ratio": float(config["VALIDATION_SPLIT"]),
            "n_jobs": int(config["N_JOBS"]),
            "seed": int(config["RANDOM_STATE"]),
            "log_file_name": str(result_dir / "flaml.log"),
            "verbose": 2,
        }
        estimadores = config.get("FLAML_ESTIMATOR_LIST")
        if estimadores:
            fit_kwargs["estimator_list"] = estimadores

        modelo.fit(**fit_kwargs)
        try:
            modelo.pickle(str(model_path))
        except Exception:
            joblib.dump(modelo, model_path)

    y_pred = modelo.predict(X_test)
    y_proba = (
        modelo.predict_proba(X_test)
        if hasattr(modelo, "predict_proba")
        else None
    )

    leaderboard = pd.DataFrame(
        [{
            "best_estimator": getattr(modelo, "best_estimator", None),
            "best_config": getattr(modelo, "best_config", None),
            "best_loss": getattr(modelo, "best_loss", None),
            "best_iteration": getattr(modelo, "best_iteration", None),
        }]
    )
    return y_pred, y_proba, leaderboard, {"search_metric": "macro_f1"}


def executar_lazypredict(X_train, X_test, y_train, y_test, config, result_dir):
    from lazypredict.Supervised import LazyClassifier

    def macro_f1_metric(y_true, y_pred):
        return f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        )

    modo = str(config.get("LAZYPREDICT_MODE", "stable")).lower()
    classifiers = "all"

    if modo == "stable":
        from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
        from sklearn.ensemble import (
            AdaBoostClassifier,
            ExtraTreesClassifier,
            RandomForestClassifier,
        )
        from sklearn.linear_model import (
            LogisticRegression,
            RidgeClassifier,
            SGDClassifier,
        )
        from sklearn.naive_bayes import GaussianNB
        from sklearn.neighbors import KNeighborsClassifier
        from sklearn.tree import DecisionTreeClassifier

        classifiers = [
            LogisticRegression,
            RidgeClassifier,
            SGDClassifier,
            DecisionTreeClassifier,
            RandomForestClassifier,
            ExtraTreesClassifier,
            AdaBoostClassifier,
            KNeighborsClassifier,
            GaussianNB,
            LinearDiscriminantAnalysis,
        ]

    clf = LazyClassifier(
        verbose=0,
        ignore_warnings=True,
        custom_metric=macro_f1_metric,
        predictions=True,
        random_state=int(config["RANDOM_STATE"]),
        classifiers=classifiers,
    )

    modelos, predicoes = clf.fit(
        X_train,
        X_test,
        y_train,
        y_test,
    )

    if modelos is None or len(modelos) == 0:
        raise RuntimeError("LazyPredict não retornou modelos.")
    if predicoes is None or len(predicoes) == 0:
        raise RuntimeError("LazyPredict não retornou predições.")

    modelos_export = modelos.reset_index()
    primeira_coluna = modelos_export.columns[0]
    modelos_export = modelos_export.rename(
        columns={primeira_coluna: "Model"}
    )

    coluna_macro = next(
        (
            c for c in modelos.columns
            if "macro_f1_metric" in str(c).lower()
            or "custom" in str(c).lower()
        ),
        None,
    )
    if coluna_macro:
        melhor_nome = str(
            pd.to_numeric(
                modelos[coluna_macro],
                errors="coerce",
            ).idxmax()
        )
    elif "F1 Score" in modelos.columns:
        melhor_nome = str(
            pd.to_numeric(
                modelos["F1 Score"],
                errors="coerce",
            ).idxmax()
        )
    else:
        melhor_nome = str(modelos.index[0])

    if melhor_nome in predicoes.columns:
        y_pred = predicoes[melhor_nome]
    else:
        y_pred = predicoes.iloc[:, 0]
        melhor_nome = str(predicoes.columns[0])

    modelo_obj = getattr(clf, "models", {}).get(melhor_nome)
    y_proba = None

    if modelo_obj is not None:
        try:
            joblib.dump(
                modelo_obj,
                result_dir / "modelo_lazypredict.pkl",
            )
        except Exception:
            pass

        if hasattr(modelo_obj, "predict_proba"):
            try:
                y_proba = modelo_obj.predict_proba(X_test)
            except Exception:
                y_proba = None

    return (
        y_pred,
        y_proba,
        modelos_export,
        {
            "search_metric": "Macro-F1 customizado",
            "best_model": melhor_nome,
            "mode": modo,
        },
    )


def executar_autopytorch(X_train, X_test, y_train, y_test, config, result_dir):
    from autoPyTorch.api.tabular_classification import (
        TabularClassificationTask,
    )
    from autoPyTorch.datasets.resampling_strategy import HoldoutValTypes

    label_encoder = LabelEncoder()
    y_train_enc = label_encoder.fit_transform(y_train)
    joblib.dump(label_encoder, result_dir / "label_encoder.pkl")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train).astype(
        np.float32,
        copy=False,
    )
    X_test_scaled = scaler.transform(X_test).astype(
        np.float32,
        copy=False,
    )
    joblib.dump(scaler, result_dir / "scaler.pkl")

    model_path = result_dir / "modelo_autopytorch.pkl"
    leaderboard = None

    tmp_dir = Path("/content") / f"autopytorch_tmp_{os.getpid()}"
    out_dir = Path("/content") / f"autopytorch_out_{os.getpid()}"
    shutil.rmtree(tmp_dir, ignore_errors=True)
    shutil.rmtree(out_dir, ignore_errors=True)

    if config["CARREGAR_MODELO_EXISTENTE"] and model_path.exists():
        with open(model_path, "rb") as f:
            api = pickle.load(f)
    else:
        if not config["TREINAR_SE_NAO_EXISTIR"]:
            raise FileNotFoundError(model_path)

        estrategia_validacao = (
            HoldoutValTypes.stratified_holdout_validation
            if config["STRATIFY"]
            else HoldoutValTypes.holdout_validation
        )

        api = TabularClassificationTask(
            seed=int(config["RANDOM_STATE"]),
            n_jobs=int(config["N_JOBS"]),
            n_threads=int(config.get("AUTOPYTORCH_N_THREADS", 2)),
            ensemble_size=int(
                config.get("AUTOPYTORCH_ENSEMBLE_SIZE", 5)
            ),
            ensemble_nbest=int(
                config.get("AUTOPYTORCH_ENSEMBLE_NBEST", 5)
            ),
            max_models_on_disc=int(
                config.get("AUTOPYTORCH_MAX_MODELS_ON_DISC", 10)
            ),
            temporary_directory=str(tmp_dir),
            output_directory=str(out_dir),
            delete_tmp_folder_after_terminate=False,
            delete_output_folder_after_terminate=False,
            resampling_strategy=estrategia_validacao,
            resampling_strategy_args={
                "val_share": float(config["VALIDATION_SPLIT"])
            },
        )

        func_limit = max(
            60,
            min(
                int(config.get("AUTOPYTORCH_FUNC_EVAL_SECS", 180)),
                max(60, int(config["TIME_LIMIT"]) // 3),
            ),
        )

        api.search(
            X_train=X_train_scaled,
            y_train=y_train_enc,
            dataset_name=f"CICIoT2023_{config['CENARIO']}",
            optimize_metric="f1_macro",
            budget_type="epochs",
            min_budget=int(
                config.get("AUTOPYTORCH_MIN_EPOCHS", 5)
            ),
            max_budget=int(
                config.get("AUTOPYTORCH_MAX_EPOCHS", 20)
            ),
            total_walltime_limit=int(config["TIME_LIMIT"]),
            func_eval_time_limit_secs=func_limit,
            memory_limit=int(config["AUTOPYTORCH_MEMORY_MB"]),
            enable_traditional_pipeline=bool(
                config.get(
                    "AUTOPYTORCH_ENABLE_TRADITIONAL",
                    True,
                )
            ),
        )

        try:
            api.refit(
                X_train=X_train_scaled,
                y_train=y_train_enc,
            )
        except Exception as exc:
            (result_dir / "aviso_refit_autopytorch.txt").write_text(
                repr(exc),
                encoding="utf-8",
            )

    y_pred_enc = api.predict(X_test_scaled)
    try:
        y_proba = api.predict_proba(X_test_scaled)
    except Exception:
        y_proba = None

    if not (
        config["CARREGAR_MODELO_EXISTENTE"]
        and model_path.exists()
    ):
        try:
            with open(model_path, "wb") as f:
                pickle.dump(api, f)
        except Exception as exc:
            (result_dir / "aviso_pickle_autopytorch.txt").write_text(
                repr(exc),
                encoding="utf-8",
            )

    try:
        modelos = api.show_models()
        if isinstance(modelos, dict) and modelos:
            linhas_modelos = []
            for model_id, dados_modelo in modelos.items():
                if isinstance(dados_modelo, dict):
                    linha = {"model_id": model_id, **dados_modelo}
                else:
                    linha = {"model_id": model_id, "detalhes": str(dados_modelo)}
                linhas_modelos.append(linha)
            leaderboard = pd.DataFrame(linhas_modelos)
        else:
            leaderboard = pd.DataFrame(
                [{"statistics": api.sprint_statistics()}]
            )
    except Exception:
        try:
            leaderboard = pd.DataFrame(
                [{"statistics": api.sprint_statistics()}]
            )
        except Exception:
            leaderboard = None

    info = {
        "search_metric": "f1_macro",
        "label_classes": list(label_encoder.classes_),
        "resampling_strategy": str(
            HoldoutValTypes.stratified_holdout_validation
            if config["STRATIFY"]
            else HoldoutValTypes.holdout_validation
        ),
        "temporary_directory": str(tmp_dir),
        "output_directory": str(out_dir),
    }
    return (
        y_pred_enc,
        y_proba,
        leaderboard,
        info,
        label_encoder,
    )


def executar_autokeras(X_train, X_test, y_train, config, result_dir):
    import autokeras as ak
    import keras
    import keras_tuner as kt
    import tensorflow as tf

    # Compatibilidade AutoKeras 3.0.0 + Keras 3.x:
    # o AutoKeras pode encaminhar np.int64 como quantidade de unidades,
    # enquanto algumas versões do Keras aceitam somente int nativo.
    dense_init_original = keras.layers.Dense.__init__
    if not getattr(dense_init_original, "_ciciot_units_compat", False):
        def dense_init_compat(self, units, *args, **kwargs):
            try:
                units = int(units)
            except (TypeError, ValueError):
                pass
            return dense_init_original(self, units, *args, **kwargs)

        dense_init_compat._ciciot_units_compat = True
        keras.layers.Dense.__init__ = dense_init_compat

    try:
        for gpu in tf.config.list_physical_devices("GPU"):
            tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

    keras.backend.clear_session()

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y_train)
    joblib.dump(label_encoder, result_dir / "label_encoder.pkl")

    X_fit, X_val, y_fit, y_val = train_test_split(
        X_train,
        y_encoded,
        test_size=float(config["VALIDATION_SPLIT"]),
        random_state=int(config["RANDOM_STATE"]),
        stratify=y_encoded if config["STRATIFY"] else None,
    )

    scaler = StandardScaler()
    X_fit_scaled = scaler.fit_transform(X_fit).astype(
        np.float32,
        copy=False,
    )
    X_val_scaled = scaler.transform(X_val).astype(
        np.float32,
        copy=False,
    )
    X_test_scaled = scaler.transform(X_test).astype(
        np.float32,
        copy=False,
    )
    joblib.dump(scaler, result_dir / "scaler.pkl")

    class MacroF1Callback(keras.callbacks.Callback):
        def __init__(self, x_val, y_val):
            super().__init__()
            self.x_val = x_val
            self.y_val = y_val
            self.history_macro_f1 = []

        def on_epoch_end(self, epoch, logs=None):
            logs = logs if logs is not None else {}
            pred = np.asarray(
                self.model.predict(
                    self.x_val,
                    verbose=0,
                )
            )
            if pred.ndim == 2 and pred.shape[1] > 1:
                pred_label = np.argmax(pred, axis=1)
            else:
                pred_label = (
                    pred.reshape(-1) >= 0.5
                ).astype(int)

            valor = float(
                f1_score(
                    self.y_val,
                    pred_label,
                    average="macro",
                    zero_division=0,
                )
            )
            logs["val_macro_f1"] = valor
            self.history_macro_f1.append(valor)

    model_path = result_dir / "modelo_autokeras.keras"
    tuner_dir = (
        Path("/content")
        / f"autokeras_tuner_{os.getpid()}"
    )
    shutil.rmtree(tuner_dir, ignore_errors=True)

    if config["CARREGAR_MODELO_EXISTENTE"] and model_path.exists():
        modelo = keras.models.load_model(
            model_path,
            compile=False,
        )
        macro_callback = None
    else:
        if not config["TREINAR_SE_NAO_EXISTIR"]:
            raise FileNotFoundError(model_path)

        macro_callback = MacroF1Callback(
            X_val_scaled,
            y_val,
        )

        objetivo = str(
            config.get(
                "AUTOKERAS_OBJECTIVE",
                "val_accuracy",
            )
        )
        if objetivo == "val_macro_f1":
            objective = kt.Objective(
                "val_macro_f1",
                direction="max",
            )
            monitor = "val_macro_f1"
        else:
            objective = objetivo
            monitor = objetivo

        clf = ak.StructuredDataClassifier(
            num_classes=int(len(label_encoder.classes_)),
            max_trials=int(config["AUTOKERAS_MAX_TRIALS"]),
            overwrite=True,
            seed=int(config["RANDOM_STATE"]),
            objective=objective,
            directory=str(tuner_dir),
            project_name=f"ciciot2023_{config['CENARIO']}",
        )

        callbacks = [macro_callback]
        if monitor != "val_macro_f1":
            callbacks.append(
                keras.callbacks.EarlyStopping(
                    monitor=monitor,
                    mode="max",
                    patience=int(
                        config.get(
                            "AUTOKERAS_PATIENCE",
                            3,
                        )
                    ),
                    restore_best_weights=True,
                )
            )

        clf.fit(
            X_fit_scaled,
            y_fit,
            validation_data=(X_val_scaled, y_val),
            epochs=int(config["AUTOKERAS_EPOCHS"]),
            callbacks=callbacks,
            verbose=1,
        )

        # Extrair os trials do tuner para apresentar um leaderboard real.
        linhas_trials = []
        try:
            for trial_id, trial in clf.tuner.oracle.trials.items():
                linha = {
                    "trial_id": trial_id,
                    "status": str(trial.status),
                    "score": trial.score,
                    "best_step": getattr(trial, "best_step", None),
                }
                for chave, valor in trial.hyperparameters.values.items():
                    linha[f"hp_{chave}"] = valor
                linhas_trials.append(linha)
        except Exception as exc:
            (result_dir / "aviso_leaderboard_autokeras.txt").write_text(
                repr(exc), encoding="utf-8"
            )

        leaderboard = pd.DataFrame(linhas_trials) if linhas_trials else None
        if leaderboard is not None and "score" in leaderboard.columns:
            leaderboard = leaderboard.sort_values(
                "score", ascending=False, na_position="last"
            ).reset_index(drop=True)

        modelo = clf.export_model()
        modelo.save(model_path)

    if config["CARREGAR_MODELO_EXISTENTE"] and model_path.exists():
        leaderboard_path = result_dir / "leaderboard_autokeras.csv"
        leaderboard = (
            pd.read_csv(leaderboard_path)
            if leaderboard_path.exists()
            else None
        )

    pred_raw = np.asarray(
        modelo.predict(
            X_test_scaled,
            verbose=0,
        )
    )

    if pred_raw.ndim == 2 and pred_raw.shape[1] > 1:
        y_pred_enc = np.argmax(pred_raw, axis=1)
        y_proba = pred_raw
    else:
        score = pred_raw.reshape(-1)
        y_pred_enc = (score >= 0.5).astype(int)
        y_proba = score

    info = {
        "search_metric": str(
            config.get(
                "AUTOKERAS_OBJECTIVE",
                "val_accuracy",
            )
        ),
        "comparison_metric": "Macro-F1 externo",
        "label_classes": list(label_encoder.classes_),
        "tuner_directory": str(tuner_dir),
        "val_macro_f1_history": (
            macro_callback.history_macro_f1
            if macro_callback is not None
            else None
        ),
    }
    return (
        y_pred_enc,
        y_proba,
        leaderboard,
        info,
        label_encoder,
    )


def executar_autogluon(X_train, X_test, y_train, y_test, config, result_dir):
    from autogluon.tabular import TabularPredictor

    label_col = config["LABEL_FINAL"]
    model_dir = result_dir / "modelo_autogluon"

    train_df = X_train.copy()
    train_df[label_col] = y_train.values
    test_df = X_test.copy()
    test_df[label_col] = y_test.values

    if config["CARREGAR_MODELO_EXISTENTE"] and model_dir.exists():
        predictor = TabularPredictor.load(str(model_dir))
    else:
        if not config["TREINAR_SE_NAO_EXISTIR"]:
            raise FileNotFoundError(model_dir)

        log_autogluon = result_dir / "autogluon_interno.log"

        # O tipo do problema deve acompanhar o cenário selecionado:
        # binario -> "binary"; agrupado8/multiclasse34 -> "multiclass".
        problem_type = config["PROBLEM_TYPE"]

        predictor_kwargs = {
            "label": label_col,
            "problem_type": problem_type,
            "eval_metric": config.get("EVAL_METRIC", "f1_macro"),
            "path": str(model_dir),
            "verbosity": 3,
            "log_to_file": True,
            "log_file_path": str(log_autogluon),
        }

        # positive_class só é válido e necessário no cenário binário.
        if problem_type == "binary":
            predictor_kwargs["positive_class"] = config["CLASSE_MALICIOSA"]

        predictor = TabularPredictor(**predictor_kwargs)

        # Conjunto CPU seguro e compatível com as dependências instaladas.
        # Evita que modelos opcionais sem suas bibliotecas interrompam o fit.
        hiperparametros_seguros = {
            "GBM": {},
            "XGB": {},
            "RF": {},
            "XT": {},
        }

        predictor.fit(
            train_data=train_df,
            time_limit=int(config["TIME_LIMIT"]),
            presets=config["AUTOGLUON_PRESETS"],
            hyperparameters=hiperparametros_seguros,
            num_cpus=max(1, int(config["N_JOBS"])),
            num_gpus=int(
                config.get("AUTOGLUON_NUM_GPUS", 0)
            ),
        )

    y_pred = predictor.predict(X_test)
    try:
        y_proba = predictor.predict_proba(X_test)
    except Exception:
        y_proba = None

    try:
        leaderboard = predictor.leaderboard(
            test_df,
            silent=True,
        )
    except Exception:
        leaderboard = predictor.leaderboard(
            silent=True,
        )

    try:
        best_model = predictor.model_best
    except Exception:
        try:
            best_model = predictor.get_model_best()
        except Exception:
            best_model = None

    info = {
        "search_metric": config.get("EVAL_METRIC", "f1_macro"),
        "best_model": best_model,
        "problem_type": config["PROBLEM_TYPE"],
        "num_classes": int(config["NUM_CLASSES"]),
        "classes": list(config["CLASSES_FINAIS"]),
        "hyperparameters": ["GBM", "XGB", "RF", "XT"],
        "internal_log": str(result_dir / "autogluon_interno.log"),
    }
    return y_pred, y_proba, leaderboard, info


def executar_framework(framework: str, config: dict[str, Any]) -> None:
    framework = framework.lower().strip()
    result_dir = Path(config["RUN_RESULTS_DIR"]) / framework
    limpar_diretorio_se_solicitado(result_dir, config["APAGAR_MODELO_ANTERIOR"])

    inicio = time.time()
    status = {
        "framework": framework,
        "status": "running",
        "inicio_epoch": inicio,
        "python": sys.version,
        "executable": sys.executable,
    }
    salvar_json(status, result_dir / "status.json")

    X_train = X_test = y_train = y_test = None
    try:
        X_train, X_test, y_train, y_test, prep_info = preparar_dados_numericos(
            Path(config["TRAIN_PARQUET"]),
            Path(config["TEST_PARQUET"]),
            config["LABEL_FINAL"],
        )
        salvar_json(prep_info, result_dir / "preprocessamento.json")

        classes = config["CLASSES_FINAIS"]
        label_encoder = None

        if framework == "sklearn":
            y_pred, y_proba, leaderboard, info = executar_sklearn(
                X_train, X_test, y_train, config, result_dir
            )
        elif framework == "pycaret":
            y_pred, y_proba, leaderboard, info = executar_pycaret(
                X_train, X_test, y_train, config, result_dir
            )
        elif framework == "tpot":
            y_pred, y_proba, leaderboard, info = executar_tpot(
                X_train, X_test, y_train, config, result_dir
            )
        elif framework == "h2o":
            y_pred, y_proba, leaderboard, info = executar_h2o(
                X_train, X_test, y_train, config, result_dir
            )
        elif framework == "autosklearn":
            y_pred, y_proba, leaderboard, info = executar_autosklearn(
                X_train, X_test, y_train, config, result_dir
            )
        elif framework == "flaml":
            y_pred, y_proba, leaderboard, info = executar_flaml(
                X_train, X_test, y_train, config, result_dir
            )
        elif framework == "lazypredict":
            y_pred, y_proba, leaderboard, info = executar_lazypredict(
                X_train, X_test, y_train, y_test, config, result_dir
            )
        elif framework == "autopytorch":
            y_pred, y_proba, leaderboard, info, label_encoder = executar_autopytorch(
                X_train, X_test, y_train, y_test, config, result_dir
            )
        elif framework == "autokeras":
            y_pred, y_proba, leaderboard, info, label_encoder = executar_autokeras(
                X_train, X_test, y_train, config, result_dir
            )
        elif framework == "autogluon":
            y_pred, y_proba, leaderboard, info = executar_autogluon(
                X_train, X_test, y_train, y_test, config, result_dir
            )
        else:
            raise ValueError(f"Framework inválido: {framework}")

        metricas = avaliar_e_salvar(
            framework=framework,
            y_test=y_test,
            y_pred=y_pred,
            y_proba=y_proba,
            result_dir=result_dir,
            classes=classes,
            classe_positiva=(
                config["CLASSE_MALICIOSA"]
                if config["PROBLEM_TYPE"] == "binary"
                else None
            ),
            leaderboard=leaderboard,
            extra_info=info,
            label_encoder=label_encoder,
        )

        fim = time.time()
        status.update(
            {
                "status": "success",
                "fim_epoch": fim,
                "duracao_segundos": fim - inicio,
                "metricas": metricas,
            }
        )
        salvar_json(status, result_dir / "status.json")

    except Exception as exc:
        fim = time.time()
        status.update(
            {
                "status": "error",
                "fim_epoch": fim,
                "duracao_segundos": fim - inicio,
                "erro": repr(exc),
                "traceback": traceback.format_exc(),
            }
        )
        salvar_json(status, result_dir / "status.json")
        with open(result_dir / "erro.txt", "w", encoding="utf-8") as f:
            f.write(status["traceback"])
        raise
    finally:
        try:
            import h2o
            h2o.cluster().shutdown(prompt=False)
        except Exception:
            pass
        del X_train, X_test, y_train, y_test
        gc.collect()


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--framework", required=True)
    parser.add_argument("--config", required=True)
    args = parser.parse_args()

    config = carregar_config(args.config)
    executar_framework(args.framework, config)


if __name__ == "__main__":
    main()

'''


## 9. Geração da configuração de cada cenário

In [14]:
def gerar_configuracao_do_runner():

    # ============================================================
    # 8. GERAR O EXECUTOR E O ARQUIVO DE CONFIGURAÇÃO
    # ============================================================

    import json

    RUNNER_SCRIPT.write_text(
        RUNNER_CODE,
        encoding="utf-8",
    )

    CONFIG_EXECUCAO = {
        "DATASET_DIR": str(DATASET_DIR),
        "RESULTS_DIR": str(RESULTS_DIR),
        "RUN_RESULTS_DIR": str(RUN_RESULTS_DIR),
        "TRAIN_PARQUET": str(TRAIN_PARQUET),
        "TEST_PARQUET": str(TEST_PARQUET),

        "CENARIO": CENARIO,
        "CLASSES_FINAIS": CLASSES_FINAIS,
        "NUM_CLASSES": NUM_CLASSES,
        "PROBLEM_TYPE": PROBLEM_TYPE,
        "EXECUTAR_FRAMEWORK": EXECUTAR_FRAMEWORK,
        "FRAMEWORKS": FRAMEWORKS,



        "LABEL_ORIGEM": LABEL_ORIGEM,
        "LABEL_FINAL": LABEL_FINAL,
        "CLASSE_BENIGNA": CLASSE_BENIGNA,
        "CLASSE_MALICIOSA": CLASSE_MALICIOSA,
        "CLASSES_BINARIAS": CLASSES_BINARIAS,

        "CHUNKSIZE": CHUNKSIZE,
        "MAX_FILES": MAX_FILES,
        "MAX_ROWS_PER_FILE": MAX_ROWS_PER_FILE,
        "MAX_ROWS_PER_CLASS": MAX_ROWS_PER_CLASS,
        "IGNORE_MERGED_FILES": IGNORE_MERGED_FILES,

        "TEST_SIZE": TEST_SIZE,
        "VALIDATION_SPLIT": VALIDATION_SPLIT,
        "RANDOM_STATE": RANDOM_STATE,
        "STRATIFY": STRATIFY,

        "OBJECTIVE": OBJECTIVE,
        "OPTIMIZE_METRIC": OPTIMIZE_METRIC,
        "EVAL_METRIC": EVAL_METRIC,

        "CARREGAR_MODELO_EXISTENTE": CARREGAR_MODELO_EXISTENTE,
        "TREINAR_SE_NAO_EXISTIR": TREINAR_SE_NAO_EXISTIR,
        "APAGAR_MODELO_ANTERIOR": APAGAR_MODELO_ANTERIOR,

        "N_JOBS": N_JOBS,
        "TIME_LIMIT": TIME_LIMIT,

        "PYCARET_FOLDS": PYCARET_FOLDS,
        "PYCARET_INCLUDE_MODELS": PYCARET_INCLUDE_MODELS,

        "TPOT_GENERATIONS": TPOT_GENERATIONS,
        "TPOT_POPULATION_SIZE": TPOT_POPULATION_SIZE,
        "TPOT_CV": TPOT_CV,
        "TPOT_MAX_EVAL_TIME_MINS": TPOT_MAX_EVAL_TIME_MINS,
        "TPOT_EARLY_STOP": TPOT_EARLY_STOP,

        "H2O_NTHREADS": H2O_NTHREADS,
        "H2O_MAX_MEM_SIZE": H2O_MAX_MEM_SIZE,
        "H2O_MAX_MODELS": H2O_MAX_MODELS,
        "H2O_NFOLDS": H2O_NFOLDS,
        "H2O_MAX_RUNTIME_PER_MODEL": H2O_MAX_RUNTIME_PER_MODEL,
        "H2O_STOPPING_ROUNDS": H2O_STOPPING_ROUNDS,
        "H2O_STOPPING_TOLERANCE": H2O_STOPPING_TOLERANCE,
        "H2O_KEEP_CV_MODELS": H2O_KEEP_CV_MODELS,
        "H2O_KEEP_CV_PREDICTIONS": H2O_KEEP_CV_PREDICTIONS,
        "H2O_KEEP_CV_FOLD_ASSIGNMENT": H2O_KEEP_CV_FOLD_ASSIGNMENT,

        "AUTOSKLEARN_MEMORY_MB": AUTOSKLEARN_MEMORY_MB,
        "AUTOSKLEARN_PER_RUN_SECS": AUTOSKLEARN_PER_RUN_SECS,
        "AUTOSKLEARN_ENSEMBLE_SIZE": AUTOSKLEARN_ENSEMBLE_SIZE,
        "AUTOSKLEARN_ENSEMBLE_NBEST": AUTOSKLEARN_ENSEMBLE_NBEST,
        "AUTOSKLEARN_MAX_MODELS_ON_DISC": AUTOSKLEARN_MAX_MODELS_ON_DISC,

        "FLAML_ESTIMATOR_LIST": FLAML_ESTIMATOR_LIST,

        "LAZYPREDICT_MODE": LAZYPREDICT_MODE,

        "AUTOPYTORCH_MEMORY_MB": AUTOPYTORCH_MEMORY_MB,
        "AUTOPYTORCH_N_THREADS": AUTOPYTORCH_N_THREADS,
        "AUTOPYTORCH_ENSEMBLE_SIZE": AUTOPYTORCH_ENSEMBLE_SIZE,
        "AUTOPYTORCH_ENSEMBLE_NBEST": AUTOPYTORCH_ENSEMBLE_NBEST,
        "AUTOPYTORCH_MAX_MODELS_ON_DISC": AUTOPYTORCH_MAX_MODELS_ON_DISC,
        "AUTOPYTORCH_FUNC_EVAL_SECS": AUTOPYTORCH_FUNC_EVAL_SECS,
        "AUTOPYTORCH_MIN_EPOCHS": AUTOPYTORCH_MIN_EPOCHS,
        "AUTOPYTORCH_MAX_EPOCHS": AUTOPYTORCH_MAX_EPOCHS,
        "AUTOPYTORCH_ENABLE_TRADITIONAL": AUTOPYTORCH_ENABLE_TRADITIONAL,

        "AUTOKERAS_MAX_TRIALS": AUTOKERAS_MAX_TRIALS,
        "AUTOKERAS_EPOCHS": AUTOKERAS_EPOCHS,
        "AUTOKERAS_PATIENCE": AUTOKERAS_PATIENCE,
        "AUTOKERAS_OBJECTIVE": AUTOKERAS_OBJECTIVE,

        "AUTOGLUON_PRESETS": AUTOGLUON_PRESETS,
        "AUTOGLUON_NUM_GPUS": AUTOGLUON_NUM_GPUS,
    }

    CONFIG_JSON.write_text(
        json.dumps(
            CONFIG_EXECUCAO,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    # Verificações antes de iniciar treinamentos longos.
    compile(
        RUNNER_SCRIPT.read_text(encoding="utf-8"),
        str(RUNNER_SCRIPT),
        "exec",
    )

    for caminho in [TRAIN_PARQUET, TEST_PARQUET]:
        if not caminho.exists():
            raise FileNotFoundError(
                f"Split preparado não encontrado: {caminho}"
            )

    print("Executor criado:", RUNNER_SCRIPT)
    print("Configuração criada:", CONFIG_JSON)
    print("Sintaxe do executor: OK")


## 10. Execução sequencial dos modelos

In [15]:
def executar_modelos_do_cenario(executar_framework=None):

    # ============================================================
    # 9. EXECUTAR OS FRAMEWORKS SEQUENCIALMENTE
    # ============================================================

    import gc
    import json
    import time
    import traceback

    instalar_micromamba()

    selecao = executar_framework
    if selecao is None:
        selecao = globals().get("EXECUTAR_FRAMEWORK", "todos")
    selecao = str(selecao).lower().strip()

    if selecao == "todos":
        frameworks_selecionados = list(FRAMEWORKS)
    else:
        if selecao not in FRAMEWORKS:
            raise ValueError(
                f"Framework inválido: {selecao}. "
                f"Opções: {FRAMEWORKS} ou 'todos'."
            )
        frameworks_selecionados = [selecao]

    print("Frameworks selecionados:", frameworks_selecionados)

    status_geral = {}
    inicio_geral = time.time()

    for indice, framework in enumerate(
        frameworks_selecionados,
        start=1,
    ):
        print("\n" + "=" * 100)
        print(
            f"[{indice}/{len(frameworks_selecionados)}] "
            f"FRAMEWORK: {framework.upper()}"
        )
        print("=" * 100)

        framework_dir = RUN_RESULTS_DIR / framework
        framework_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        inicio_framework = time.time()
        sucesso = False

        try:
            criar_ambiente(framework)
            executar_no_ambiente(
                framework,
                CONFIG_JSON,
            )
            sucesso = True

            runner_status_path = (
                framework_dir
                / "status.json"
            )
            runner_status = {}

            if runner_status_path.exists():
                runner_status = json.loads(
                    runner_status_path.read_text(
                        encoding="utf-8"
                    )
                )

            status_geral[framework] = {
                "status": "success",
                "duracao_segundos": (
                    time.time()
                    - inicio_framework
                ),
                "runner_status": runner_status,
                "log_execucao": str(
                    LOGS_DIR
                    / f"{framework}_execucao.log"
                ),
            }

        except Exception as exc:
            runner_status_path = (
                framework_dir
                / "status.json"
            )
            erro_path = (
                framework_dir
                / "erro.txt"
            )

            runner_status = None
            erro_runner = None

            if runner_status_path.exists():
                try:
                    runner_status = json.loads(
                        runner_status_path.read_text(
                            encoding="utf-8"
                        )
                    )
                except Exception:
                    runner_status = None

            if erro_path.exists():
                erro_runner = erro_path.read_text(
                    encoding="utf-8",
                    errors="replace",
                )

            duracao_erro = time.time() - inicio_framework
            traceback_erro = traceback.format_exc()
            saida_comando = getattr(exc, "output", None)
            log_comando = getattr(exc, "log_path", None)
            saida_final = (
                str(saida_comando)[-20000:]
                if saida_comando
                else None
            )
            status_geral[framework] = {
                "status": "error",
                "duracao_segundos": duracao_erro,
                "erro_orquestrador": repr(exc),
                "runner_status": runner_status,
                "erro_runner": erro_runner,
                "saida_final_comando": saida_final,
                "log_comando_que_falhou": log_comando,
                "traceback_orquestrador": traceback_erro,
                "log_execucao": str(
                    LOGS_DIR
                    / f"{framework}_execucao.log"
                ),
            }

            # Se a falha ocorreu antes de iniciar o runner (instalação/verificação
            # do ambiente), cria status.json para a consolidação não marcar o
            # framework incorretamente como "not_executed".
            if not runner_status_path.exists():
                runner_status_path.write_text(
                    json.dumps(
                        {
                            "framework": framework,
                            "status": "error",
                            "duracao_segundos": duracao_erro,
                            "erro": repr(exc),
                            "saida_final_comando": saida_final,
                            "log_comando_que_falhou": log_comando,
                            "traceback": traceback_erro,
                            "origem_erro": "orquestrador_ou_ambiente",
                        },
                        indent=2,
                        ensure_ascii=False,
                        default=str,
                    ),
                    encoding="utf-8",
                )

            print(
                f"\nFalha em {framework}: {exc}"
            )

            if saida_final:
                print("\nÚltimas linhas do comando que falhou:")
                print("-" * 100)
                print(saida_final)

            if log_comando:
                print(f"\nLog completo: {log_comando}")

            if erro_runner:
                print("\nErro interno do framework:")
                print("-" * 100)
                print(erro_runner[-12000:])

            if not CONTINUAR_APOS_ERRO:
                raise

        finally:
            if sucesso and REMOVER_AMBIENTE_APOS_SUCESSO:
                remover_ambiente(framework)

            elif (
                not sucesso
                and not MANTER_AMBIENTE_SE_ERRO
            ):
                remover_ambiente(framework)

            elif not sucesso:
                print(
                    f"Ambiente {nome_ambiente(framework)} "
                    "mantido para diagnóstico."
                )

            gc.collect()

    status_geral["_resumo"] = {
        "inicio_epoch": inicio_geral,
        "fim_epoch": time.time(),
        "duracao_total_segundos": (
            time.time()
            - inicio_geral
        ),
        "frameworks_solicitados": (
            frameworks_selecionados
        ),
    }

    status_path = (
        RUN_RESULTS_DIR
        / "status_geral.json"
    )

    status_path.write_text(
        json.dumps(
            status_geral,
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        encoding="utf-8",
    )

    print("\nExecução concluída.")
    print("Status geral:", status_path)
    print(
        json.dumps(
            status_geral,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
    )


## 11. Métricas, gráficos e ranking por Macro-F1

In [16]:
def consolidar_resultados_do_cenario():

    # ============================================================
    # 10. CONSOLIDAR RESULTADOS E RANKING MACRO-F1
    # ============================================================

    import json
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    status_geral_path = RUN_RESULTS_DIR / "status_geral.json"
    status_geral_execucao = {}
    if status_geral_path.exists():
        status_geral_execucao = json.loads(
            status_geral_path.read_text(encoding="utf-8")
        )

    linhas_resumo = []

    for framework in FRAMEWORKS:
        framework_dir = RUN_RESULTS_DIR / framework
        metricas_path = framework_dir / f"metricas_{framework}.csv"
        status_path = framework_dir / "status.json"

        linha = {
            "Framework": framework,
            "Status": "not_executed",
            "Duração (s)": np.nan,
            "Erro": None,
        }

        # Prioriza o status detalhado do runner; usa status_geral como fallback
        # para falhas de instalação/verificação do ambiente.
        status = None
        if status_path.exists():
            try:
                status = json.loads(status_path.read_text(encoding="utf-8"))
            except Exception as exc:
                linha["Erro"] = f"Falha ao ler status.json: {exc!r}"

        status_orquestrador = status_geral_execucao.get(framework, {})

        if status:
            linha["Status"] = status.get("status", "unknown")
            linha["Duração (s)"] = status.get("duracao_segundos", np.nan)
            linha["Erro"] = status.get("erro")
        elif status_orquestrador:
            linha["Status"] = status_orquestrador.get("status", "unknown")
            linha["Duração (s)"] = status_orquestrador.get(
                "duracao_segundos", np.nan
            )
            linha["Erro"] = (
                status_orquestrador.get("erro_runner")
                or status_orquestrador.get("erro_orquestrador")
            )

        if metricas_path.exists():
            try:
                df_m = pd.read_csv(metricas_path)
                metricas_framework = (
                    df_m.dropna(subset=["Métrica"])
                    .drop_duplicates(subset=["Métrica"], keep="first")
                    .set_index("Métrica")["Valor"]
                    .to_dict()
                )
                linha.update(metricas_framework)
                # A existência das métricas comprova que a avaliação terminou.
                if linha["Status"] in {"not_executed", "unknown", "running"}:
                    linha["Status"] = "success"
            except Exception as exc:
                mensagem = f"Falha ao consolidar métricas: {exc!r}"
                linha["Erro"] = (
                    f"{linha['Erro']} | {mensagem}" if linha["Erro"] else mensagem
                )

        linhas_resumo.append(linha)

    resumo = pd.DataFrame(linhas_resumo)

    # Converter métricas para números e organizar as colunas principais.
    colunas_identificacao = ["Framework", "Status", "Duração (s)", "Erro"]
    for coluna in resumo.columns:
        if coluna not in colunas_identificacao:
            resumo[coluna] = pd.to_numeric(resumo[coluna], errors="coerce")

    colunas_metricas_preferidas = [
        "Accuracy",
        "Balanced Accuracy",
        "Precision Macro",
        "Recall Macro",
        "Macro-F1",
        "Weighted-F1",
        "MCC",
        "Precision Malicious",
        "Recall Malicious",
        "F1 Malicious",
        "ROC-AUC",
        "Average Precision",
    ]
    colunas_metricas = [
        c for c in colunas_metricas_preferidas if c in resumo.columns
    ]
    colunas_extras = [
        c for c in resumo.columns
        if c not in colunas_identificacao + colunas_metricas
    ]
    resumo = resumo[colunas_identificacao + colunas_metricas + colunas_extras]

    # Sucessos primeiro, ordenados por Macro-F1; erros permanecem visíveis.
    resumo["_ordem_status"] = resumo["Status"].map(
        {"success": 0, "error": 1, "not_executed": 2}
    ).fillna(3)
    if "Macro-F1" in resumo.columns:
        resumo = resumo.sort_values(
            ["_ordem_status", "Macro-F1"],
            ascending=[True, False],
            na_position="last",
        )
    else:
        resumo = resumo.sort_values("_ordem_status")
    resumo = resumo.drop(columns="_ordem_status").reset_index(drop=True)

    resumo_csv = RUN_RESULTS_DIR / "resumo_comparativo_frameworks.csv"
    resumo.to_csv(resumo_csv, index=False)

    # Tabela PNG com todos os frameworks, inclusive falhas diagnosticadas.
    resumo_plot = resumo.copy()
    if "Erro" in resumo_plot.columns:
        resumo_plot["Erro"] = resumo_plot["Erro"].fillna("").astype(str).str.replace(
            "\n", " ", regex=False
        ).str.slice(0, 90)
    fig, ax = plt.subplots(
        figsize=(22, max(5, 0.65 * (len(resumo_plot) + 3)))
    )
    ax.axis("off")
    tabela_plot = resumo_plot.round(6)
    tab = ax.table(
        cellText=tabela_plot.values,
        colLabels=tabela_plot.columns,
        cellLoc="center",
        loc="center",
    )
    tab.auto_set_font_size(False)
    tab.set_fontsize(6.5)
    tab.scale(1, 1.3)
    ax.set_title(
        "Resumo comparativo dos frameworks AutoML — CICIoT2023",
        pad=12,
    )
    fig.tight_layout()
    fig.savefig(
        RUN_RESULTS_DIR / "resumo_comparativo_frameworks.png",
        dpi=250,
        bbox_inches="tight",
    )
    plt.close(fig)

    # Ranking principal apenas com frameworks que produziram Macro-F1 válido.
    if "Macro-F1" in resumo.columns:
        ranking = (
            resumo.loc[
                resumo["Macro-F1"].notna(),
                ["Framework", "Macro-F1"],
            ]
            .sort_values("Macro-F1", ascending=False)
            .reset_index(drop=True)
        )
        ranking.to_csv(RUN_RESULTS_DIR / "ranking_macro_f1.csv", index=False)

        if not ranking.empty:
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.barh(ranking["Framework"], ranking["Macro-F1"])
            ax.invert_yaxis()
            ax.set_xlabel("Macro-F1")
            ax.set_title("Ranking Macro-F1 — CICIoT2023")
            for i, valor in enumerate(ranking["Macro-F1"]):
                ax.text(valor, i, f" {valor:.4f}", va="center")
            fig.tight_layout()
            fig.savefig(
                RUN_RESULTS_DIR / "ranking_macro_f1.png",
                dpi=250,
                bbox_inches="tight",
            )
            plt.close(fig)
        else:
            print("Nenhum framework produziu Macro-F1 válido nesta execução.")

    status_df = resumo[["Framework", "Status", "Duração (s)", "Erro"]].copy()
    status_df.to_csv(
        RUN_RESULTS_DIR / "resumo_status_frameworks.csv",
        index=False,
    )

    display(resumo)
    print("Resultados consolidados em:", RUN_RESULTS_DIR)
    print("Resumo CSV:", resumo_csv)


## 12. Execução sequencial completa

In [ ]:
# ============================================================
# 12. EXECUÇÃO MESTRA: CENÁRIOS → MODELOS → MÉTRICAS
# ============================================================

import json
import time
import traceback
from pathlib import Path

# Garante que MICROMAMBA seja um objeto Path, caso tenha sido redefinido como string.
MICROMAMBA = Path(MICROMAMBA)

instalar_micromamba()

status_cenarios = {}
inicio_total = time.time()

for indice_cenario, nome_cenario in enumerate(CENARIOS, start=1):
    print("\n" + "#" * 100)
    print(f"[{indice_cenario}/{len(CENARIOS)}] INICIANDO {nome_cenario.upper()}")
    print("#" * 100)
    inicio_cenario = time.time()

    try:
        configurar_cenario(nome_cenario)
        preparar_e_validar_cenario()
        gerar_configuracao_do_runner()
        executar_modelos_do_cenario(EXECUTAR_FRAMEWORK)
        consolidar_resultados_do_cenario()
        # O orquestrador pode continuar mesmo quando um framework falha.
        # Por isso, o status do cenário deve refletir os resultados dos modelos.
        status_geral_path = RUN_RESULTS_DIR / "status_geral.json"
        if not status_geral_path.exists():
            raise FileNotFoundError(
                f"Status dos modelos não encontrado: {status_geral_path}"
            )

        status_modelos = json.loads(
            status_geral_path.read_text(encoding="utf-8")
        )
        resultados = [
            dados.get("status")
            for framework, dados in status_modelos.items()
            if (
                not framework.startswith("_")
                and isinstance(dados, dict)
                and "status" in dados
            )
        ]

        if resultados and all(
            status == "success" for status in resultados
        ):
            status_cenario = "success"
        elif any(status == "success" for status in resultados):
            status_cenario = "partial_success"
        else:
            status_cenario = "error"

        status_cenarios[nome_cenario] = {
            "status": status_cenario,
            "status_modelos": resultados,
            "duracao_segundos": time.time() - inicio_cenario,
            "diretorio": str(RUN_RESULTS_DIR),
        }
    except Exception as exc:
        status_cenarios[nome_cenario] = {
            "status": "error",
            "duracao_segundos": time.time() - inicio_cenario,
            "erro": repr(exc),
            "traceback": traceback.format_exc(),
            "diretorio": str(RUN_RESULTS_DIR),
        }
        print(f"Falha no cenário {nome_cenario}: {exc}")
        if not CONTINUAR_APOS_ERRO:
            raise

ROOT_RUN_DIR = RESULTS_DIR / RUN_ID_GERAL
ROOT_RUN_DIR.mkdir(parents=True, exist_ok=True)
(ROOT_RUN_DIR / "status_cenarios.json").write_text(
    json.dumps(status_cenarios, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# Consolidação geral dos três cenários.
quadros = []
for nome_cenario in CENARIOS:
    caminho = ROOT_RUN_DIR / nome_cenario / "resumo_comparativo_frameworks.csv"
    if caminho.exists():
        quadro = pd.read_csv(caminho)
        quadro.insert(0, "Cenário", nome_cenario)
        quadros.append(quadro)

if quadros:
    geral = pd.concat(quadros, ignore_index=True)
    geral.to_csv(ROOT_RUN_DIR / "metricas_todos_cenarios_modelos.csv", index=False)

    ranking_geral = geral.dropna(subset=["Macro-F1"]).sort_values(
        ["Cenário", "Macro-F1"], ascending=[True, False]
    )
    ranking_geral.to_csv(ROOT_RUN_DIR / "ranking_macro_f1_todos_cenarios.csv", index=False)

    fig, axes = plt.subplots(
        nrows=len(CENARIOS), ncols=1,
        figsize=(12, 5 * len(CENARIOS)), squeeze=False
    )
    for ax, nome_cenario in zip(axes.ravel(), CENARIOS):
        dados = ranking_geral[ranking_geral["Cenário"] == nome_cenario]
        ax.barh(dados["Framework"], dados["Macro-F1"])
        ax.invert_yaxis()
        ax.set_xlim(0, 1)
        ax.set_title(f"Ranking Macro-F1 — {nome_cenario}")
        ax.set_xlabel("Macro-F1")
    fig.tight_layout()
    fig.savefig(
        ROOT_RUN_DIR / "ranking_macro_f1_todos_cenarios.png",
        dpi=300, bbox_inches="tight"
    )
    plt.close(fig)
    display(ranking_geral)

print("\nPipeline completo.")
print("Resultados:", ROOT_RUN_DIR)
print("Duração total (s):", round(time.time() - inicio_total, 2))

Micromamba já instalado: /content/bin/micromamba

####################################################################################################
[1/3] INICIANDO BINARIO
####################################################################################################

CENÁRIO: binario | CLASSES: 2 | ALVO: label_binary
Arquivos selecionados: 309
[1/309] Backdoor_Malware.pcap.csv
[2/309] BenignTraffic.pcap.csv
[3/309] BenignTraffic1.pcap.csv
[4/309] BenignTraffic2.pcap.csv
[5/309] BenignTraffic3.pcap.csv
[6/309] BrowserHijacking.pcap.csv
[7/309] CommandInjection.pcap.csv
[8/309] DDoS-ACK_Fragmentation.pcap.csv
[9/309] DDoS-ACK_Fragmentation1.pcap.csv
[10/309] DDoS-ACK_Fragmentation10.pcap.csv
[11/309] DDoS-ACK_Fragmentation11.pcap.csv
[12/309] DDoS-ACK_Fragmentation12.pcap.csv
[13/309] DDoS-ACK_Fragmentation2.pcap.csv
[14/309] DDoS-ACK_Fragmentation3.pcap.csv
[15/309] DDoS-ACK_Fragmentation4.pcap.csv
[16/309] DDoS-ACK_Fragmentation5.pcap.csv
[17/309] DDoS-ACK_Fragmentation6.pcap.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

A saída de streaming foi truncada nas últimas 5000 linhas.
----------------------------------------------------------------------------------------------------

$ /content/bin/micromamba run -n ciciot_flaml python -m pip install --no-cache-dir numpy==1.26.4 pandas==2.1.4 scikit-learn==1.5.2 matplotlib==3.8.4 pyarrow==15.0.2 joblib psutil flaml[automl]==2.6.0 xgboost==2.1.4 lightgbm==4.6.0
----------------------------------------------------------------------------------------------------
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 228.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 210.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 235.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 230.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.3/38.3 MB 181.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 MB 292.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Saídas geradas

Para cada combinação cenário–framework são salvos, quando aplicáveis:

- métricas em CSV e tabela em PNG;
- matriz de confusão em CSV e PNG;
- relatório de classificação em CSV e PNG;
- predições do conjunto de teste;
- leaderboard, histórico ou resumo do modelo;
- modelo treinado, configuração, status, erro e log de execução.

Na pasta raiz da execução são gerados:

- `metricas_todos_cenarios_modelos.csv`;
- `ranking_macro_f1_todos_cenarios.csv`;
- `ranking_macro_f1_todos_cenarios.png`;
- `status_cenarios.json`.

### Critério científico

O **Macro-F1** é o critério principal porque atribui o mesmo peso a cada classe,
independentemente do número de amostras. Accuracy, Balanced Accuracy, Precision
Macro, Recall Macro, Weighted-F1, MCC e ROC-AUC permanecem como métricas
complementares. O conjunto de teste é usado apenas na avaliação final.
